# Stage 08 — diverse spectrum-to-SMILES inference
Reuse the Stage 07 v2 checkpoint **without retraining**. Run beam 32 / return 32,
plus 50 nucleus samples (p=0.9, T=0.8), at molecule level on two isolated GPU
subprocesses. Only batch size changes on OOM. GPU allocator cap: 11 GB/process;
host RSS budgets: parent 2 GB + two 3.5 GB workers. No soft time cutoff.

Attach clean train/test, splits, Stage 07 checkpoint/vocabulary/config and Stage 05
element arrays with ID tables. Optional: formula DB and Stage 06 candidate tables.
**Before Run All, place the original Stage 02 `evalkit.py` at
`/kaggle/working/evalkit.py`. It is read only from that exact path.**
All other Python modules are embedded; RDKit/torch must be preinstalled. Internet OFF.
This checkpoint uses SMILES, so SELFIES and its wheel are not needed.

Outputs under `/kaggle/working/artifacts/spec2smiles_infer`: all three `cand_gen_*`
tables, `submission_gen.csv`, formula ranks, query diagnostics, metrics and manifest.
Scores are higher-is-better. Runtime is measured, including an initial 200-query
projection; stage-10 estimates cover this generator component, not other stages.
`SMOKE_TEST=True` runs a small synthetic CPU verification without attached inputs.

In [ ]:
CFG = dict(seed=42, SMOKE_TEST=False, work='/kaggle/working',
    out='/kaggle/working/artifacts/spec2smiles_infer',
    cache='/kaggle/working/spec2smiles_infer_cache', resume=True,
    beam=32, num_return=32, samples=50, top_p=0.9, temperature=0.8,
    length_penalty=0.7, max_length=128, batch_size=4, ppm=10., relax_ppm=50.,
    gpu_memory_gb=11., parent_ram_gb=2., worker_ram_gb=3.5,
    time_budget_s=12*3600, stage10_generator_budget_s=8*3600,
    alpha_grid=[0., 0.1, 0.25, 0.5, 1.],
    formula_weight_grid=[0., 0.1, 0.25], prior_weight=0.15, presence_weight=0.5,
    formula_max_counts={}, queries_per_part=100,
    elements_order=['C','H','N','O','S','P','F','Cl','Br','I'])

In [ ]:
import gc, hashlib, importlib, json, os, subprocess, sys, time
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import psutil
import torch
from rdkit import Chem
from tqdm.auto import tqdm
START = time.perf_counter()
WORK, OUT, CACHE = [Path(CFG[k]) for k in ('work', 'out', 'cache')]
for directory in (WORK, OUT, CACHE):
    directory.mkdir(parents=True, exist_ok=True)
print(dict(torch=torch.__version__, rdkit=Chem.rdBase.rdkitVersion,
    numpy=np.__version__, pandas=pd.__version__, gpu_count=torch.cuda.device_count()))
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, round(p.total_memory / 1e9, 2), 'GB')
assert 25 <= CFG['beam'] <= 50 and 25 <= CFG['num_return'] <= 50
assert CFG['samples'] == 50 and CFG['relax_ppm'] == 50

## Embedded model and inference code
The saved v2 model has **no trained formula-prefix/no-formula tokens**. Its native
predicted-element conditioning is retained for one beam pass and one sampling pass.
Formula enumeration/ranking still runs independently, and its score can rerank
generated structures (weight selected on val_model). No invented tokens or retraining.
A different formula-prefix checkpoint fails explicitly instead of silently ignoring it.

Spectra are pooled by a sqrt(peak-count)-weighted mean of valid encoded positions,
pooled embeddings and condition vectors. Formula search covers the train-observed
maximum count domain, optionally enlarged with `formula_max_counts`, with train-only
1st–99th percentile element/C ratios. This empirical domain can exclude novel formulas;
formula recall reports that limitation. DB frequencies never set chemical bounds.

In [ ]:
module_path = WORK / 'spec2fp.py'
module_path.write_text('', encoding='utf-8')

In [ ]:
fragment = (
'"""spec2fp.py - Spectrum-to-Fingerprint model, tensorization, and inference.\n'
'\n'
'Defines the Spec2FP Transformer, peak tensorization, and predict() API.\n'
'Import ONLY from /kaggle/working/.\n'
'"""\n'
'import json, math, sys, warnings, gc\n'
'from pathlib import Path\n'
'from typing import Dict, List, Optional, Tuple, Union, Any\n'
'import numpy as np\n'
'import pandas as pd\n'
'import torch\n'
'import torch.nn as nn\n'
'import torch.nn.functional as F\n'
'\n'
'ADDUCT_VOCAB = [\n'
'    "<pad>", "[M+H]+", "[M-H]-", "[M+Na]+", "[M+NH4]+", "[M+K]+",\n'
'    "[M+H-H2O]+", "[M+FA-H]-", "[M+Cl]-", "[M]+", "[M-H2O+H]+",\n'
'    "[M-H2O-H]-", "[M+2H]2+", "[M-2H]2-", "[M+HAc-H]-",\n'
'    "[M+2Na-H]+", "[M+CH2O2-H]-", "<unk>",\n'
']\n'
'ADDUCT2IDX = {a: i for i, a in enumerate(ADDUCT_VOCAB)}\n'
'INST_VOCAB = [\n'
'    "<pad>", "timsTOF", "qtof", "orbitrap", "FTICR", "QqQ",\n'
'    "ion_trap", "None/Unknown", "<unk>",\n'
']\n'
'INST2IDX = {a: i for i, a in enumerate(INST_VOCAB)}\n'
'POLARITY_VOCAB = ["<pad>", "positive", "negative", "<unk>"]\n'
'POLARITY2IDX = {a: i for i, a in enumerate(POLARITY_VOCAB)}\n'
'ELEMENTS_ORDER = ["C", "H", "N", "O", "S", "P", "F", "Cl", "Br", "I"]\n'
'N_ELEMENTS = len(ELEMENTS_ORDER)\n'
'\n'
'\n'
'def parse_peak_array(val: Any) -> np.ndarray:\n'
'    """Robustly parse peak m/z or intensity into a float64 numpy array."""\n'
'    if isinstance(val, np.ndarray):\n'
'        return val.astype(np.float64)\n'
'    if isinstance(val, (list, tuple)):\n'
'        return np.asarray(val, dtype=np.float64)\n'
'    if isinstance(val, str):\n'
'        try:\n'
'            return np.asarray(json.loads(val), dtype=np.float64)\n'
'        except Exception:\n'
'            pass\n'
'    try:\n'
'        return np.asarray(val, dtype=np.float64)\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'    except Exception:\n'
'        return np.zeros(0, dtype=np.float64)\n'
'\n'
'\n'
'def sinusoidal_embed(values: np.ndarray, dim: int = 64) -> np.ndarray:\n'
'    """Multi-scale sinusoidal encoding resolving 0.005 Da mass defects up to 2000 Da."""\n'
'    freqs = np.logspace(\n'
'        math.log10(2 * math.pi / 2000.0),\n'
'        math.log10(2 * math.pi / 0.005),\n'
'        dim // 2, dtype=np.float32)\n'
'    angles = np.outer(values.astype(np.float32), freqs)\n'
'    return np.concatenate([np.sin(angles), np.cos(angles)], axis=-1)\n'
'\n'
'\n'
'def tensorize_spectrum(\n'
'    mz: Any, inten: Any, precursor_mz: float,\n'
'    adduct: Any, ce: float, instrument: Any, polarity: Any,\n'
'    top_k: int = 192, mz_dim: int = 64,\n'
') -> Dict[str, np.ndarray]:\n'
'    """Convert a single spectrum into model-ready numpy arrays."""\n'
'    mz = parse_peak_array(mz)\n'
'    inten = parse_peak_array(inten)\n'
'    mask_valid = mz <= precursor_mz + 5.0\n'
'    mz, inten = mz[mask_valid], inten[mask_valid]\n'
'    if len(mz) == 0:\n'
'        mz = np.array([precursor_mz], dtype=np.float64)\n'
'        inten = np.array([1.0], dtype=np.float64)\n'
'    max_int = max(inten.max(), 1e-12)\n'
'    inten = np.sqrt(inten / max_int)\n'
'    if len(mz) > top_k:\n'
'        idx = np.argsort(inten)[-top_k:]\n'
'        idx = idx[np.argsort(mz[idx])]\n'
'        mz, inten = mz[idx], inten[idx]\n'
'    n = len(mz)\n'
'    mz_emb = sinusoidal_embed(mz, mz_dim)\n'
'    nl = precursor_mz - mz\n'
'    nl_emb = sinusoidal_embed(nl, mz_dim)\n'
'    log_int = np.log1p(inten).astype(np.float32).reshape(-1, 1)\n'
'    peak_feat = np.concatenate([mz_emb, log_int, nl_emb], axis=1).astype(np.float32)\n'
'    adduct_idx = int(adduct) if isinstance(adduct, (int, np.integer)) else ADDUCT2IDX.get(str(adduct).strip(), ADDUCT2IDX["<unk>"])\n'
'    inst_idx = int(instrument) if isinstance(instrument, (int, np.integer)) else INST2IDX.get(str(instrument).strip(), INST2IDX["<unk>"])\n'
'    pol_idx = int(polarity) if isinstance(polarity, (int, np.integer)) else POLARITY2IDX.get(str(polarity).strip().lower(), POLARITY2IDX["<unk>"])\n'
'    ce_val = float(ce) if not (ce is None or (isinstance(ce, float) and math.isnan(ce))) else 0.0\n'
'    return {\n'
'        "peak_feat": peak_feat, "n_peaks": n, "mz": mz,\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'        "adduct_idx": adduct_idx, "inst_idx": inst_idx, "pol_idx": pol_idx,\n'
'        "ce_val": np.float32(ce_val), "prec_mz": np.float32(precursor_mz),\n'
'    }\n'
'\n'
'\n'
'def collate_spectra(batch: list, top_k: int = 192, feat_dim: int = 129):\n'
'    """Collate list of tensorize_spectrum outputs into batched tensors."""\n'
'    B = len(batch)\n'
'    max_len = top_k + 3\n'
'    peaks = torch.zeros(B, max_len, feat_dim, dtype=torch.float32)\n'
'    mask = torch.ones(B, max_len, dtype=torch.bool)\n'
'    adducts = torch.zeros(B, dtype=torch.long)\n'
'    instruments = torch.zeros(B, dtype=torch.long)\n'
'    polarities = torch.zeros(B, dtype=torch.long)\n'
'    ces = torch.zeros(B, dtype=torch.float32)\n'
'    prec_mzs = torch.zeros(B, dtype=torch.float32)\n'
'    for i, item in enumerate(batch):\n'
'        n = item["n_peaks"]\n'
'        peaks[i, :n] = torch.from_numpy(item["peak_feat"][:n])\n'
'        mask[i, :n] = False\n'
'        adducts[i] = item["adduct_idx"]\n'
'        instruments[i] = item["inst_idx"]\n'
'        polarities[i] = item["pol_idx"]\n'
'        ces[i] = float(item["ce_val"])\n'
'        prec_mzs[i] = float(item["prec_mz"])\n'
'    return {\n'
'        "peaks": peaks, "pad_mask": mask,\n'
'        "adduct": adducts, "instrument": instruments, "polarity": polarities,\n'
'        "ce": ces, "prec_mz": prec_mzs,\n'
'    }\n'
'\n'
'\n'
'class Spec2FPModel(nn.Module):\n'
'    """Transformer encoder mapping MS/MS spectrum to molecular fingerprint."""\n'
'\n'
'    def __init__(self, cfg: dict):\n'
'        super().__init__()\n'
'        self.cfg = cfg\n'
'        d, n_layers, n_heads = cfg.get("d_model", 320), cfg.get("n_layers", 5), cfg.get("n_heads", 8)\n'
'        dropout, feat_dim = cfg.get("dropout", 0.1), cfg.get("feat_dim", 129)\n'
'        fp_bits, maccs_bits = cfg.get("fp_bits", 4096), cfg.get("maccs_bits", 167)\n'
'        n_elements = cfg.get("n_elements", N_ELEMENTS)\n'
'        self.peak_proj = nn.Sequential(nn.Linear(feat_dim, d), nn.GELU(), nn.LayerNorm(d), nn.Dropout(dropout))\n'
'        self.cls_token = nn.Parameter(torch.randn(1, 1, d) * 0.02)\n'
'        self.adduct_emb = nn.Embedding(len(ADDUCT_VOCAB), d, padding_idx=0)\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'        self.inst_emb = nn.Embedding(len(INST_VOCAB), d, padding_idx=0)\n'
'        self.pol_emb = nn.Embedding(len(POLARITY_VOCAB), d, padding_idx=0)\n'
'        self.ce_proj = nn.Sequential(nn.Linear(1, d), nn.GELU())\n'
'        self.prec_proj = nn.Sequential(nn.Linear(1, d), nn.GELU())\n'
'        enc_layer = nn.TransformerEncoderLayer(\n'
'            d_model=d, nhead=n_heads, dim_feedforward=d * 4,\n'
'            dropout=dropout, activation="gelu", batch_first=True, norm_first=True)\n'
'        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)\n'
'        self.norm = nn.LayerNorm(d)\n'
'        self.fp_head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, fp_bits))\n'
'        self.maccs_head = nn.Sequential(nn.Linear(d, d // 2), nn.GELU(), nn.Linear(d // 2, maccs_bits))\n'
'        self.heavy_atom_head = nn.Linear(d, 1)\n'
'        self.ring_count_head = nn.Linear(d, 1)\n'
'        self.elem_presence_head = nn.Linear(d, n_elements)\n'
'        self.elem_count_head = nn.Linear(d, n_elements)\n'
'\n'
'    def forward(self, peaks, pad_mask, adduct, instrument, polarity, ce, prec_mz):\n'
'        B = peaks.size(0)\n'
'        x = self.peak_proj(peaks)\n'
'        cls = self.cls_token.expand(B, -1, -1)\n'
'        meta1 = (\n'
'            self.adduct_emb(adduct)\n'
'            + self.ce_proj(ce.unsqueeze(-1) / 100.0)\n'
'            + self.prec_proj(prec_mz.unsqueeze(-1) / 1000.0)\n'
'        )\n'
'        meta2 = self.inst_emb(instrument) + self.pol_emb(polarity)\n'
'        meta = torch.stack([meta1, meta2], dim=1)\n'
'        x = torch.cat([cls, meta, x], dim=1)\n'
'        cls_meta_mask = torch.zeros(B, 3, dtype=torch.bool, device=pad_mask.device)\n'
'        full_mask = torch.cat([cls_meta_mask, pad_mask], dim=1)\n'
'        x = self.encoder(x, src_key_padding_mask=full_mask)\n'
'        cls_out = x[:, 0]\n'
'        valid_tok = (~full_mask).float().unsqueeze(-1)\n'
'        mean_out = (x * valid_tok).sum(dim=1) / valid_tok.sum(dim=1).clamp(min=1.0)\n'
'        x = self.norm(cls_out + mean_out)\n'
'        return {\n'
'            "fp_logits": self.fp_head(x), "maccs_logits": self.maccs_head(x),\n'
'            "heavy_atoms": self.heavy_atom_head(x).squeeze(-1),\n'
'            "ring_count": self.ring_count_head(x).squeeze(-1),\n'
'            "elem_presence": self.elem_presence_head(x),\n'
'            "elem_count": self.elem_count_head(x),\n'
'        }\n'
'\n'
'\n'
'def load_model(config_path: str, weights_path: str, device: str = "cpu"):\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'    """Load Spec2FP model from config + weights."""\n'
'    with open(config_path) as f:\n'
'        cfg = json.load(f)\n'
'    model = Spec2FPModel(cfg["model"])\n'
'    sd = torch.load(weights_path, map_location=device, weights_only=False)\n'
'    if "ema_state_dict" in sd:\n'
'        model.load_state_dict({k.replace("module.", ""): v for k, v in sd["ema_state_dict"].items()})\n'
'    elif "model_state_dict" in sd:\n'
'        model.load_state_dict({k.replace("module.", ""): v for k, v in sd["model_state_dict"].items()})\n'
'    else:\n'
'        model.load_state_dict({k.replace("module.", ""): v for k, v in sd.items()})\n'
'    pw_path = Path(weights_path).parent / "spec2fp_pos_weights.npy"\n'
'    if pw_path.is_file():\n'
'        try:\n'
'            model.pos_weights = np.load(pw_path)\n'
'        except Exception:\n'
'            pass\n'
'    model.eval()\n'
'    return model.to(device), cfg\n'
'\n'
'\n'
'@torch.no_grad()\n'
'def predict_full(model, clean_df, device="cuda", batch_size=128, top_k=192, mz_dim=64, pos_weights=None):\n'
'    """Predict Morgan-4096, MACCS-167, and element counts with streaming molecule-level pooling."""\n'
'    raw_model = model.module if hasattr(model, "module") else model\n'
'    raw_model.eval()\n'
'    pw = pos_weights if pos_weights is not None else getattr(raw_model, "pos_weights", None)\n'
'    pw_shift = 0.5 * torch.log(torch.as_tensor(pw, device=device, dtype=torch.float32).clamp(min=1.0)) if pw is not None else None\n'
'    feat_dim, n_rows = 2 * mz_dim + 1, len(clean_df)\n'
'    if n_rows == 0:\n'
'        return (np.zeros((0, 4096), dtype=np.float16), np.zeros((0, 167), dtype=np.float16),\n'
'                np.zeros((0, N_ELEMENTS), dtype=np.float32), [])\n'
'\n'
'    mol_ids_arr = clean_df["molecule_id"].to_numpy().astype(str)\n'
'    unique_mols, inv_indices = np.unique(mol_ids_arr, return_inverse=True)\n'
'    n_unique = len(unique_mols)\n'
'    fp_sum = np.zeros((n_unique, 4096), dtype=np.float32)\n'
'    maccs_sum = np.zeros((n_unique, 167), dtype=np.float32)\n'
'    elem_sum = np.zeros((n_unique, N_ELEMENTS), dtype=np.float32)\n'
'    counts = np.zeros(n_unique, dtype=np.float32)\n'
'\n'
'    mz_col = clean_df["mz"].to_numpy()\n'
'    int_col = clean_df["inten"].to_numpy()\n'
'    pmz_col = clean_df["precursor_mz"].to_numpy(dtype=np.float32)\n'
'    add_col = clean_df["adduct"].to_numpy() if "adduct" in clean_df.columns else np.array(["[M+H]+"] * n_rows)\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'    ce_col = pd.to_numeric(clean_df["ce"], errors="coerce").fillna(0.0).to_numpy(dtype=np.float32) if "ce" in clean_df.columns else np.zeros(n_rows, dtype=np.float32)\n'
'    inst_col = clean_df["instrument"].to_numpy() if "instrument" in clean_df.columns else np.array(["unknown"] * n_rows)\n'
'    pol_col = clean_df["polarity"].to_numpy() if "polarity" in clean_df.columns else np.array(["positive"] * n_rows)\n'
'    dev_type, is_cuda = ("cuda" if "cuda" in str(device) else "cpu"), ("cuda" in str(device))\n'
'\n'
'    for s_idx in range(0, n_rows, batch_size):\n'
'        e_idx = min(s_idx + batch_size, n_rows)\n'
'        items = [\n'
'            tensorize_spectrum(\n'
'                mz_col[r], int_col[r], float(pmz_col[r]),\n'
'                str(add_col[r]), float(ce_col[r]),\n'
'                str(inst_col[r]), str(pol_col[r]), top_k=top_k, mz_dim=mz_dim)\n'
'            for r in range(s_idx, e_idx)\n'
'        ]\n'
'        bd = collate_spectra(items, top_k=top_k, feat_dim=feat_dim)\n'
'        bd = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in bd.items()}\n'
'        with torch.amp.autocast(device_type=dev_type, dtype=torch.float16, enabled=is_cuda):\n'
'            out = raw_model(**bd)\n'
'        fp_logits = out["fp_logits"] - pw_shift if pw_shift is not None else out["fp_logits"]\n'
'        batch_fp = torch.sigmoid(fp_logits).float().cpu().numpy()\n'
'        batch_maccs = torch.sigmoid(out["maccs_logits"]).float().cpu().numpy()\n'
'        batch_elem = out["elem_count"].float().cpu().numpy()\n'
'        batch_inv = inv_indices[s_idx:e_idx]\n'
'        np.add.at(fp_sum, batch_inv, batch_fp)\n'
'        np.add.at(maccs_sum, batch_inv, batch_maccs)\n'
'        np.add.at(elem_sum, batch_inv, batch_elem)\n'
'        np.add.at(counts, batch_inv, 1.0)\n'
'\n'
'    counts_col = np.maximum(counts, 1.0)[:, None]\n'
'    return ((fp_sum / counts_col).astype(np.float16),\n'
'            (maccs_sum / counts_col).astype(np.float16),\n'
'            (elem_sum / counts_col).astype(np.float32), unique_mols.tolist())\n'
'\n'
'\n'
'@torch.no_grad()\n'
'def predict(model, clean_df, device="cuda", batch_size=128, top_k=192, mz_dim=64, pos_weights=None, return_maccs=False):\n'
'    """Predict fingerprints with optional MACCS return for backward-compatibility."""\n'
'    fp_pred, maccs_pred, elem_pred, mol_ids = predict_full(\n'
'        model, clean_df, device=device, batch_size=batch_size, top_k=top_k, mz_dim=mz_dim, pos_weights=pos_weights\n'
'    )\n'
'    if return_maccs:\n'
'        return fp_pred, maccs_pred, elem_pred, mol_ids\n'
'    return fp_pred, elem_pred, mol_ids\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
module_path = WORK / 'spec2smiles.py'
module_path.write_text('', encoding='utf-8')

In [ ]:
fragment = (
'"""Stage 07: transferable spectral encoder, 180-dim substructure conditioning, and Self-Spec2FP reranked SMILES decoder."""\n'
'import copy\n'
'import json\n'
'import math\n'
'import re\n'
'from itertools import islice\n'
'from pathlib import Path\n'
'\n'
'import numpy as np\n'
'import pandas as pd\n'
'import torch\n'
'from torch import nn\n'
'from rdkit import Chem\n'
'from rdkit.Chem import rdMolDescriptors\n'
'import evalkit\n'
'from spec2fp import (Spec2FPModel, ELEMENTS_ORDER, tensorize_spectrum,\n'
'                     collate_spectra, parse_peak_array)\n'
'\n'
"ENCODER_NAMES = ('peak_proj', 'cls_token', 'adduct_emb', 'inst_emb',\n"
"                 'pol_emb', 'ce_proj', 'prec_proj', 'encoder', 'norm')\n"
"AUX_HEAD_NAMES = ('fp_head', 'maccs_head', 'heavy_atom_head',\n"
"                  'ring_count_head', 'elem_presence_head', 'elem_count_head')\n"
"COLUMNS = ['molecule_id', 'smiles', 'ik14', 'fast_key', 'score',\n"
"           'log_prob', 'nb_ll', 'cos_sim', 'tan_sim', 'maccs_cos', 'maccs_tan',\n"
"           'elem_dist', 'ppm_err', 'rank', 'source']\n"
'\n'
"SMILES_PATTERN = r'(\\[[^\\]]+]|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p|\\(|\\)|\\.|=|#|-|\\+|\\\\|\\/|:|~|@|\\?|>|\\*|\\$|\\%[0-9]{2}|[0-9])'\n"
'\n'
'\n'
'class SmilesTokenizer:\n'
'    def __init__(self, vocab=None, max_length=128):\n'
"        self.vocab = vocab or dict(zip(('<pad>', '<bos>', '<eos>', '<unk>'), range(4)))\n"
'        self.max_length = max_length\n'
'        self.pattern = re.compile(SMILES_PATTERN)\n'
'        self.inverse = {v: k for k, v in self.vocab.items()}\n'
'\n'
'    def tokens(self, smiles):\n'
'        tokens = self.pattern.findall(smiles)\n'
"        if ''.join(tokens) != smiles:\n"
"            raise ValueError(f'Unrecognized SMILES syntax: {smiles}')\n"
'        return tokens\n'
'\n'
'    def fit(self, smiles):\n'
'        found = set()\n'
'        for smi in smiles:\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'            found.update(self.tokens(smi))\n'
"        found.update(list('0123456789') + [f'%{i:02d}' for i in range(10, 30)])\n"
'        self.vocab.update({s: i + 4 for i, s in enumerate(sorted(found))})\n'
'        self.inverse = {v: k for k, v in self.vocab.items()}\n'
'        return self\n'
'\n'
'    def encode(self, smiles):\n'
'        tokens = self.tokens(smiles)\n'
'        if len(tokens) + 2 > self.max_length:\n'
"            raise ValueError('SMILES exceeds sequence budget; never truncate molecules')\n"
'        ids = [1] + [self.vocab.get(t, 3) for t in tokens] + [2]\n'
'        return ids + [0] * (self.max_length - len(ids))\n'
'\n'
'    def decode(self, ids):\n'
'        out = []\n'
'        for idx in ids:\n'
'            if int(idx) == 2:\n'
'                break\n'
'            if int(idx) not in (0, 1):\n'
'                out.append(self.inverse[int(idx)])\n'
"        return ''.join(out)\n"
'\n'
'    def save(self, path):\n'
'        Path(path).write_text(json.dumps(dict(vocab=self.vocab,\n'
'                                             max_length=self.max_length), indent=2))\n'
'\n'
'    @classmethod\n'
'    def load(cls, path):\n'
'        return cls(**json.loads(Path(path).read_text()))\n'
'\n'
'\n'
'def mol_elem_counts(mol):\n'
'    counts = np.zeros(len(ELEMENTS_ORDER), dtype=np.float32)\n'
'    mol_h = Chem.AddHs(mol)\n'
'    for atom in mol_h.GetAtoms():\n'
'        sym = atom.GetSymbol()\n'
'        if sym in ELEMENTS_ORDER:\n'
'            counts[ELEMENTS_ORDER.index(sym)] += 1.0\n'
'    return counts\n'
'\n'
'\n'
'class Spec2SmilesModel(nn.Module):\n'
'    def __init__(self, cfg):\n'
'        super().__init__()\n'
'        self.cfg = cfg\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
"        base = Spec2FPModel(cfg['encoder'])\n"
'        for name in ENCODER_NAMES:\n'
'            setattr(self, name, getattr(base, name))\n'
'        # Retain all pretrained Stage 05b auxiliary heads (frozen) for conditioning & reranking\n'
'        for name in AUX_HEAD_NAMES:\n'
'            head = getattr(base, name)\n'
'            head.requires_grad_(False)\n'
'            setattr(self, name, head)\n'
'\n'
"        enc_dim = cfg['encoder']['d_model']\n"
'        self.memory_proj = nn.Linear(enc_dim, 320) if enc_dim != 320 else nn.Identity()\n'
'        # 180-dim conditioning: neutral_mass(1) + elem_counts(10) + maccs_probs(167) + heavy(1) + rings(1)\n'
'        self.condition = nn.Sequential(\n'
'            nn.Linear(180, 320), nn.GELU(), nn.LayerNorm(320),\n'
'            nn.Linear(320, 320), nn.GELU(), nn.LayerNorm(320)\n'
'        )\n'
'        self.pooled_proj = nn.Sequential(nn.Linear(enc_dim, 320), nn.LayerNorm(320))\n'
"        self.token_emb = nn.Embedding(cfg['vocab_size'], 320, padding_idx=0)\n"
'        nn.init.normal_(self.token_emb.weight, std=0.02)\n'
'        with torch.no_grad():\n'
'            self.token_emb.weight[0].zero_()\n'
'        layer = nn.TransformerDecoderLayer(320, 8, 1280, 0.1,\n'
'                                           batch_first=True, norm_first=True)\n'
"        self.decoder = nn.TransformerDecoder(layer, cfg.get('decoder_layers', 4), norm=nn.LayerNorm(320))\n"
"        self.output = nn.Linear(320, cfg['vocab_size'], bias=False)\n"
'        self.output.weight = self.token_emb.weight\n'
'        pos = torch.arange(128).float().unsqueeze(1)\n'
'        angles = pos * torch.exp(torch.arange(0, 320, 2).float() * (-math.log(10000)/320))\n'
'        pe = torch.zeros(128, 320)\n'
'        pe[:, 0::2], pe[:, 1::2] = angles.sin(), angles.cos()\n'
"        self.register_buffer('positions', pe)\n"
"        self.register_buffer('pw_shift', torch.zeros(4096))\n"
'\n'
'    def encoder_parameters(self):\n'
"        return [p for n, p in self.named_parameters() if n.split('.')[0] in ENCODER_NAMES]\n"
'\n'
'    def warm_start(self, path):\n'
"        state = torch.load(path, map_location='cpu', weights_only=False)\n"
"        state = state.get('ema_state_dict', state.get('model_state_dict', state))\n"
"        state = {k.removeprefix('module.'): v for k, v in state.items()}\n"
'        wanted = {k: v for k, v in self.state_dict().items()\n'
"                  if k.split('.')[0] in ENCODER_NAMES + AUX_HEAD_NAMES}\n"
'        missing = set(wanted) - set(state)\n'
'        if missing:\n'
"            raise ValueError(f'Incomplete pretrained encoder: {sorted(missing)}')\n"
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'        self.load_state_dict({k: state[k] for k in wanted}, strict=False)\n'
"        pw_path = Path(path).parent / 'spec2fp_pos_weights.npy'\n"
'        if pw_path.is_file():\n'
'            pw = torch.from_numpy(np.load(pw_path)).float().clamp(min=1.0)\n'
'            self.pw_shift.copy_(0.5 * torch.log(pw))\n'
'\n'
'    def encode(self, peaks, pad_mask, adduct, instrument, polarity, ce, prec_mz, return_fp=False):\n'
'        b = len(peaks)\n'
'        meta1 = self.adduct_emb(adduct) + self.ce_proj(ce[:, None]/100)\n'
'        meta1 = meta1 + self.prec_proj(prec_mz[:, None]/1000)\n'
'        meta2 = self.inst_emb(instrument) + self.pol_emb(polarity)\n'
'        x = torch.cat((self.cls_token.expand(b, -1, -1),\n'
'                       torch.stack((meta1, meta2), 1), self.peak_proj(peaks)), 1)\n'
'        mask = torch.cat((pad_mask.new_zeros((b, 3)), pad_mask), 1)\n'
'        x = self.encoder(x, src_key_padding_mask=mask)\n'
'        valid = (~mask).unsqueeze(-1)\n'
'        pooled = self.norm(x[:, 0] + (x*valid).sum(1)/valid.sum(1).clamp_min(1))\n'
'        # Keep AUX_HEAD_NAMES frozen, but compute outside torch.no_grad() without .detach()\n'
'        # so gradients from self.condition flow back into pooled to refine encoder representations\n'
'        counts = self.elem_count_head(pooled).clamp(0, 200)\n'
'        maccs_probs = torch.sigmoid(self.maccs_head(pooled))\n'
'        heavy = self.heavy_atom_head(pooled).clamp(1, 150) / 100.0\n'
'        rings = self.ring_count_head(pooled).clamp(0, 20) / 10.0\n'
'        cond_vec = torch.cat((counts / 50.0, maccs_probs, heavy, rings), dim=-1)  # 179 dims\n'
'        if return_fp:\n'
'            with torch.no_grad():\n'
'                fp_probs = torch.sigmoid(self.fp_head(pooled) - self.pw_shift).detach()\n'
'            return self.memory_proj(self.norm(x)), mask, pooled, cond_vec, counts.detach(), fp_probs, maccs_probs.detach()\n'
'        return self.memory_proj(self.norm(x)), mask, pooled, cond_vec\n'
'\n'
'    def conditioned_memory(self, memory, mask, neutral_mass, pooled, cond_vec):\n'
'        full_cond = torch.cat((neutral_mass[:, None] / 1000.0, cond_vec), dim=-1)  # 180 dims\n'
'        cond_tok = self.condition(full_cond) + self.pooled_proj(pooled)\n'
'        return torch.cat((cond_tok[:, None], memory), 1), torch.cat((mask[:, :1]*False, mask), 1)\n'
'\n'
'    def decode(self, tokens, memory, mask):\n'
'        seq_len = tokens.shape[1]\n'
'        x = self.token_emb(tokens)*math.sqrt(320) + self.positions[:seq_len]\n'
'        causal = torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool).triu(1)\n'
'        tgt_pad = tokens.eq(0)\n'
'        tgt_mask = tgt_pad if tgt_pad.any() else None\n'
'        mem_mask = mask if mask.any() else None\n'
'        x = self.decoder(x, memory, tgt_mask=causal, tgt_key_padding_mask=tgt_mask,\n'
'                         memory_key_padding_mask=mem_mask)\n'
'        return self.output(x)\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'\n'
'    def forward(self, tokens, neutral_mass, **spectra):\n'
'        memory, mask, pooled, cond_vec = self.encode(**spectra)\n'
'        memory, mask = self.conditioned_memory(memory, mask, neutral_mass, pooled, cond_vec)\n'
'        return self.decode(tokens, memory, mask)\n'
'\n'
'\n'
'def spectrum_item(row, cfg):\n'
'    return tensorize_spectrum(row.mz, row.inten, float(row.precursor_mz),\n'
'                             row.adduct, float(row.ce), row.instrument, row.polarity,\n'
"                             top_k=cfg['top_k_peaks'], mz_dim=cfg['mz_dim'])\n"
'\n'
'\n'
'def record_dtype(top_k):\n'
"    return np.dtype([('mz', 'f4', (top_k,)), ('inten', 'f4', (top_k,)),\n"
"                     ('n', 'i2'), ('prec', 'f4'), ('mass', 'f4'), ('ce', 'f4'),\n"
"                     ('adduct', 'i2'), ('inst', 'i2'), ('pol', 'i2'), ('target', 'i4')])\n"
'\n'
'\n'
'class SpectrumDataset(torch.utils.data.Dataset):\n'
'    """Raw top peaks on disk; supports K-variant precomputed SMILES augmentation."""\n'
'    def __init__(self, path, cfg, tokenizer, smiles, tokens, augment=False, tokens_aug=None):\n'
'        self.cfg, self.tokenizer = cfg, tokenizer\n'
'        self.smiles, self.tokens, self.augment = smiles, tokens, augment\n'
"        self.aug_prob = cfg.get('aug_prob', 0.65)\n"
'        self.tokens_aug = tokens[None, ...] if tokens_aug is None else tokens_aug\n'
'        if self.tokens_aug.ndim == 2:\n'
'            self.tokens_aug = self.tokens_aug[None, ...]\n'
"        self.rows = np.memmap(path, mode='r', dtype=record_dtype(cfg['top_k_peaks']))\n"
'\n'
'    def __len__(self):\n'
'        return len(self.rows)\n'
'\n'
'    def __getitem__(self, idx):\n'
'        r = self.rows[idx]\n'
"        t_idx = int(r['target'])\n"
'        if self.augment and np.random.random() < self.aug_prob:\n'
'            k = np.random.randint(0, self.tokens_aug.shape[0])\n'
'            target = self.tokens_aug[k, t_idx].astype(np.int64)\n'
'        else:\n'
'            target = self.tokens[t_idx].astype(np.int64)\n'
"        item = tensorize_spectrum(r['mz'][:r['n']], r['inten'][:r['n']], r['prec'],\n"
"                                  int(r['adduct']), r['ce'], int(r['inst']), int(r['pol']),\n"
"                                  self.cfg['top_k_peaks'], self.cfg['mz_dim'])\n"
"        return item, float(r['mass']), target\n"
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'\n'
'    def collate(self, rows):\n'
'        items, masses, tokens = zip(*rows)\n'
"        batch = collate_spectra(items, self.cfg['top_k_peaks'], 2*self.cfg['mz_dim']+1)\n"
"        batch['neutral_mass'] = torch.tensor(masses, dtype=torch.float32)\n"
'        return batch, torch.from_numpy(np.stack(tokens))\n'
'\n'
'\n'
'def append_records(frame, stream, cfg, target_map):\n'
'    from spec2fp import ADDUCT2IDX, INST2IDX, POLARITY2IDX\n'
"    records = np.zeros(len(frame), dtype=record_dtype(cfg['top_k_peaks']))\n"
'    for i, row in enumerate(frame.itertuples(index=False)):\n'
'        if not (math.isfinite(row.neutral_mass) and row.neutral_mass > 0\n'
'                and math.isfinite(row.precursor_mz) and row.precursor_mz > 0):\n'
"            raise ValueError('Training spectra require finite positive precursor/neutral masses')\n"
'        mz, inten = parse_peak_array(row.mz), parse_peak_array(row.inten)\n'
'        valid = np.isfinite(mz) & np.isfinite(inten) & (inten >= 0)\n'
'        valid &= mz <= float(row.precursor_mz) + 5\n'
'        mz, inten = mz[valid], inten[valid]\n'
"        idx = np.argsort(inten)[-cfg['top_k_peaks']:]\n"
'        idx = idx[np.argsort(mz[idx])]\n'
'        n = len(idx)\n'
"        records[i]['mz'][:n], records[i]['inten'][:n] = mz[idx], inten[idx]\n"
"        records[i]['n'], records[i]['prec'] = n, row.precursor_mz\n"
"        records[i]['mass'], records[i]['ce'] = row.neutral_mass, row.ce\n"
"        records[i]['adduct'] = ADDUCT2IDX.get(str(row.adduct).strip(), ADDUCT2IDX['<unk>'])\n"
"        records[i]['inst'] = INST2IDX.get(str(row.instrument).strip(), INST2IDX['<unk>'])\n"
"        records[i]['pol'] = POLARITY2IDX.get(str(row.polarity).strip().lower(), POLARITY2IDX['<unk>'])\n"
"        records[i]['target'] = target_map[str(row.molecule_id)]\n"
'    records.tofile(stream)\n'
'\n'
'\n'
'class EMA:\n'
'    def __init__(self, model, decay=0.999):\n'
'        self.model = copy.deepcopy(model).eval().requires_grad_(False)\n'
'        self.decay = decay\n'
'\n'
'    @torch.no_grad()\n'
'    def update(self, model):\n'
'        for dst, src in zip(self.model.parameters(), model.parameters()):\n'
'            dst.lerp_(src, 1-self.decay)\n'
'\n'
'\n'
'def run_epoch(model, loader, device, optimizer=None, scaler=None, ema=None, scheduler=None, label_smoothing=0.05):\n'
'    training = optimizer is not None\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'    model.train(training)\n'
'    raw = model.module if isinstance(model, nn.DataParallel) else model\n'
'    for name in AUX_HEAD_NAMES:\n'
'        getattr(raw, name).eval()\n'
'    if training and not raw.cls_token.requires_grad:\n'
'        for name in ENCODER_NAMES:\n'
'            module = getattr(raw, name)\n'
'            if isinstance(module, nn.Module):\n'
'                module.eval()\n'
'    total, count = 0.0, 0\n'
'    for batch, target in loader:\n'
'        batch = {k: v.to(device) for k, v in batch.items()}\n'
'        target = target.to(device)\n'
'        with torch.set_grad_enabled(training):\n'
"            with torch.amp.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):\n"
'                logits = model(tokens=target[:, :-1], **batch)\n'
'                loss = nn.functional.cross_entropy(logits.reshape(-1, logits.shape[-1]),\n'
'                    target[:, 1:].reshape(-1), ignore_index=0, label_smoothing=label_smoothing)\n'
'            if not torch.isfinite(loss):\n'
"                raise FloatingPointError('Non-finite SMILES loss; check spectra and checkpoint')\n"
'            if training:\n'
'                optimizer.zero_grad(set_to_none=True)\n'
'                scaler.scale(loss).backward()\n'
'                scaler.unscale_(optimizer)\n'
'                nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n'
'                old_scale = scaler.get_scale()\n'
'                scaler.step(optimizer)\n'
'                scaler.update()\n'
'                if scaler.get_scale() >= old_scale:\n'
'                    ema.update(raw)\n'
'                    if scheduler is not None:\n'
'                        scheduler.step()\n'
'        n = int(target[:, 1:].ne(0).sum())\n'
'        total, count = total + float(loss.detach())*n, count+n\n'
'    if not count:\n'
"        raise ValueError('Empty training/validation loader')\n"
'    return total/count\n'
'\n'
'\n'
'@torch.no_grad()\n'
'def beam_search(model, memory, mask, beam_size=14, max_len=96, top_k=25, archive_size=64):\n'
'    """Decode B molecules together; return length-penalized completed paths."""\n'
'    if beam_size < 1 or max_len < 2 or max_len > 128 or top_k < 1:\n'
"        raise ValueError('Invalid beam size, sequence budget, or candidate count')\n"
'    b, length, dim = memory.shape\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'    k, device = beam_size, memory.device\n'
'    memory = memory[:, None].expand(-1, k, -1, -1).reshape(b*k, length, dim)\n'
'    mask = mask[:, None].expand(-1, k, -1).reshape(b*k, length)\n'
'    seq = torch.ones((b, k, 1), dtype=torch.long, device=device)\n'
'    scores = torch.full((b, k), -torch.inf, device=device)\n'
'    scores[:, 0] = 0\n'
'    archive_size = max(archive_size, top_k)\n'
'    archive_scores = torch.full((b, archive_size), -torch.inf, device=device)\n'
'    archive = torch.zeros((b, archive_size, max_len), dtype=torch.long, device=device)\n'
'    stop_idx = min(top_k - 1, archive_size - 1)\n'
'    for step in range(1, max_len):\n'
'        logits = model.decode(seq.reshape(b*k, -1), memory, mask)\n'
'        logp = logits[:, -1].float().log_softmax(-1).reshape(b, k, -1)\n'
'        logp[:, :, [0, 1, 3]] = -torch.inf\n'
'        if step == 1:\n'
'            logp[:, :, 2] = -torch.inf\n'
'        combined = logp + scores[:, :, None]\n'
'        vocab = combined.shape[-1]\n'
'        values, indices = combined.flatten(1).topk(min(2*k, k*vocab), dim=1)\n'
'        parents, tokens = indices // vocab, indices % vocab\n'
'        paths = torch.cat((seq.gather(1, parents[:, :, None].expand(-1, -1, step)),\n'
'                           tokens[:, :, None]), dim=2)\n'
'        eos_scores = values.masked_fill(tokens.ne(2), -torch.inf)\n'
'        norm_eos_scores = eos_scores / ((step + 1) ** 0.7)\n'
'        padded = nn.functional.pad(paths, (0, max_len-step-1))\n'
'        archive_scores, keep = torch.cat((archive_scores, norm_eos_scores), 1).topk(archive_size, dim=1)\n'
'        archive = torch.cat((archive, padded), 1).gather(1, keep[:, :, None].expand(-1, -1, max_len))\n'
'        scores, live = values.masked_fill(tokens.eq(2), -torch.inf).topk(k, dim=1)\n'
'        seq = paths.gather(1, live[:, :, None].expand(-1, -1, step+1))\n'
'        if step >= 20:\n'
'            live_upper = scores.max(1).values / ((step + 1) ** 0.7)\n'
'            target_thresh = archive_scores[:, stop_idx]\n'
'            done = (target_thresh > -torch.inf) & (live_upper <= target_thresh)\n'
'            scores = scores.masked_fill(done[:, None], -torch.inf)\n'
'            if not torch.isfinite(scores).any():\n'
'                break\n'
'    result = []\n'
'    for paths, values in zip(archive.cpu().tolist(), archive_scores.cpu().tolist()):\n'
'        result.append([(ids[:ids.index(2)+1], score) for ids, score in zip(paths, values)\n'
'                       if math.isfinite(score) and 2 in ids])\n'
'    return result\n'
'\n'
'\n'
'@torch.no_grad()\n'
'def generate_candidates(model, tokenizer, query_df, device, beam_size=14, top_k=25, mol_batch_size=64, max_len=96):\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'    model = model.module if isinstance(model, nn.DataParallel) else model\n'
'    model.eval()\n'
'    device = torch.device(device)\n'
'    rows = []\n'
'    if beam_size < 1 or top_k < 1 or mol_batch_size < 1:\n'
"        raise ValueError('beam_size and top_k must be positive')\n"
"    top_k_peaks = model.cfg['top_k_peaks']\n"
"    groups = iter(query_df.groupby('molecule_id', sort=False))\n"
'    while True:\n'
'        chunk = list(islice(groups, mol_batch_size))\n'
'        if not chunk:\n'
'            break\n'
'        memories, masks, masses, pred_fps, pred_maccs, pred_elems = [], [], [], [], [], []\n'
'        for molecule_id, group in chunk:\n'
'            mass = float(group.neutral_mass.median())\n'
'            if not math.isfinite(mass) or mass <= 0:\n'
"                raise ValueError(f'Invalid neutral mass for {molecule_id}')\n"
'            meta_mem_sum, pooled_sum, cond_sum, elem_sum, fp_sum, maccs_sum = None, None, None, None, None, None\n'
'            total_weight = 0.0\n'
'            peak_tokens_list, peak_intens_list, peak_mzs_list = [], [], []\n'
"            with torch.amp.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):\n"
'                for start in range(0, len(group), 32):\n'
'                    sub = group.iloc[start:start+32]\n'
'                    items = [spectrum_item(r, model.cfg) for r in sub.itertuples()]\n'
"                    batch = collate_spectra(items, top_k_peaks, 2*model.cfg['mz_dim']+1)\n"
'                    memory, mask, pooled, cond_vec, counts, fp_p, maccs_p = model.encode(\n'
'                        **{k: v.to(device) for k, v in batch.items()}, return_fp=True\n'
'                    )\n'
'                    # Weight spectra by sqrt(n_peaks) for signal-to-noise calibrated pooling\n'
'                    valid_peaks = ~mask[:, 3:]\n'
'                    w_spec = valid_peaks.sum(1).float().sqrt().clamp_min(1.0)\n'
'                    w_col = w_spec[:, None]\n'
'                    meta_mem = (memory[:, :3].float() * w_spec[:, None, None]).sum(0, keepdim=True)\n'
'                    meta_mem_sum = meta_mem if meta_mem_sum is None else meta_mem_sum + meta_mem\n'
'                    pooled_w = (pooled.float() * w_col).sum(0, keepdim=True)\n'
'                    pooled_sum = pooled_w if pooled_sum is None else pooled_sum + pooled_w\n'
'                    cond_w = (cond_vec.float() * w_col).sum(0, keepdim=True)\n'
'                    cond_sum = cond_w if cond_sum is None else cond_sum + cond_w\n'
'                    elem_w = (counts.float() * w_col).sum(0, keepdim=True)\n'
'                    elem_sum = elem_w if elem_sum is None else elem_sum + elem_w\n'
'                    fp_sum = (fp_p.float() * w_col).sum(0) if fp_sum is None else fp_sum + (fp_p.float() * w_col).sum(0)\n'
'                    maccs_sum = (maccs_p.float() * w_col).sum(0) if maccs_sum is None else maccs_sum + (maccs_p.float() * w_col).sum(0)\n'
'                    total_weight += float(w_spec.sum().item())\n'
'                    for it in items:\n'
"                        peak_mzs_list.append(torch.from_numpy(it['mz']).float().to(device))\n"
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'                    peak_tokens_list.append(memory[:, 3:][valid_peaks].float())\n'
"                    peak_intens_list.append(batch['peaks'][..., 64].to(device)[valid_peaks].float())\n"
'\n'
'                all_peak_toks = torch.cat(peak_tokens_list, dim=0)\n'
'                all_peak_ints = torch.cat(peak_intens_list, dim=0)\n'
'                all_peak_mzs = torch.cat(peak_mzs_list, dim=0) if peak_mzs_list else torch.zeros(0, device=device)\n'
'                mol_peaks = torch.zeros((1, top_k_peaks, 320), dtype=torch.float32, device=device)\n'
'                mol_peak_mask = torch.ones((1, top_k_peaks), dtype=torch.bool, device=device)\n'
'                if len(all_peak_toks) > 0:\n'
'                    sorted_idx = torch.argsort(all_peak_ints, descending=True)\n'
'                    bins = (all_peak_mzs[sorted_idx] * 100).long()\n'
'                    _, unique_pos = np.unique(bins.cpu().numpy(), return_index=True)\n'
'                    dedup_idx = sorted_idx[torch.from_numpy(np.sort(unique_pos)).to(device)]\n'
'                    take = min(top_k_peaks, len(dedup_idx))\n'
'                    chosen = dedup_idx[:take][torch.argsort(all_peak_mzs[dedup_idx[:take]])]\n'
'                    mol_peaks[0, :take] = all_peak_toks[chosen]\n'
'                    mol_peak_mask[0, :take] = False\n'
'\n'
'                w_norm = max(total_weight, 1e-6)\n'
'                mol_mem = torch.cat([meta_mem_sum / w_norm, mol_peaks], dim=1)\n'
'                mol_mask = torch.cat([torch.zeros((1, 3), dtype=torch.bool, device=device), mol_peak_mask], dim=1)\n'
'                memory, mask = model.conditioned_memory(\n'
'                    mol_mem, mol_mask, torch.tensor([mass], device=device), pooled_sum / w_norm, cond_sum / w_norm\n'
'                )\n'
'            memories.append(memory)\n'
'            masks.append(mask)\n'
'            masses.append(mass)\n'
'            pred_fps.append((fp_sum / w_norm).cpu().numpy())\n'
'            pred_maccs.append((maccs_sum / w_norm).cpu().numpy())\n'
'            pred_elems.append((elem_sum.squeeze(0) / w_norm).cpu().numpy())\n'
'\n'
"        with torch.amp.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):\n"
'            hypotheses = beam_search(model, torch.cat(memories), torch.cat(masks),\n'
'                                     beam_size=beam_size, max_len=max_len, top_k=top_k)\n'
'\n'
'        for (molecule_id, _), mass, p_fp, p_mc, p_el, paths in zip(\n'
'            chunk, masses, pred_fps, pred_maccs, pred_elems, hypotheses\n'
'        ):\n'
'            p_fp_clip = np.clip(p_fp, 1e-6, 1.0 - 1e-6)\n'
'            fp_logits = np.log(p_fp_clip) - np.log1p(-p_fp_clip)\n'
'            fp_log_not_p = float(np.sum(-np.logaddexp(0.0, fp_logits)))\n'
'            p_fp_norm, p_mc_norm = float(np.linalg.norm(p_fp)), float(np.linalg.norm(p_mc))\n'
'            p_fp_sum, p_mc_sum = float(p_fp.sum()), float(p_mc.sum())\n'
'            seen_raw_smi, pre_candidates = set(), {}\n'
'\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'            for ids, log_prob in paths:\n'
'                smi = tokenizer.decode(ids)\n'
'                if not smi or smi in seen_raw_smi:\n'
'                    continue\n'
'                seen_raw_smi.add(smi)\n'
'                raw_mol = Chem.MolFromSmiles(smi)\n'
'                if raw_mol is None or raw_mol.GetNumAtoms() == 0 or any(a.GetSymbol() not in ELEMENTS_ORDER for a in raw_mol.GetAtoms()):\n'
'                    continue\n'
'                raw_wt = float(rdMolDescriptors.CalcExactMolWt(raw_mol))\n'
"                mol = (evalkit.standardize_mol(raw_mol) or raw_mol) if (Chem.GetFormalCharge(raw_mol) != 0 or '.' in smi) else raw_mol\n"
'                mol_wt = float(rdMolDescriptors.CalcExactMolWt(mol)) if mol is not raw_mol else raw_wt\n'
'                key = evalkit.fast_key(mol)\n'
'                if not key:\n'
'                    continue\n'
'                ppm = abs(mol_wt - mass) / mass * 1e6\n'
'                mass_score = -0.5 * (min(ppm, 50.0) / 7.5) ** 2\n'
'                pre_score = log_prob + 0.80 * mass_score\n'
'                if key not in pre_candidates or pre_score > pre_candidates[key][0]:\n'
'                    pre_candidates[key] = (pre_score, log_prob, mass_score, ppm, mol, key)\n'
'\n'
'            # Two-stage candidate reranking: Top 28 unique fast_key candidates\n'
'            top28 = sorted(pre_candidates.values(), key=lambda x: -x[0])[:28]\n'
'            full_candidates = []\n'
'            for _, log_prob, mass_score, ppm, mol, key in top28:\n'
'                c_fp, c_mc = evalkit.compute_morgan_fp(mol).astype(np.float32), evalkit.compute_maccs_fp(mol).astype(np.float32)\n'
'                dot_fp, dot_mc = float(c_fp @ p_fp), float(c_mc @ p_mc)\n'
'                c_fp_sum, c_mc_sum = float(c_fp.sum()), float(c_mc.sum())\n'
'                cos_sim = dot_fp / max(math.sqrt(c_fp_sum) * p_fp_norm, 1e-8)\n'
'                tan_sim = dot_fp / max(c_fp_sum + p_fp_sum - dot_fp, 1e-8)\n'
'                nb_ll = (float(c_fp @ fp_logits) + fp_log_not_p) / 4096.0\n'
'                maccs_cos = dot_mc / max(math.sqrt(c_mc_sum) * p_mc_norm, 1e-8)\n'
'                maccs_tan = dot_mc / max(c_mc_sum + p_mc_sum - dot_mc, 1e-8)\n'
'                elem_dist = float(np.abs(mol_elem_counts(mol) - p_el).sum())\n'
'                score = (log_prob + 2.0 * nb_ll + 1.8 * cos_sim + 1.8 * tan_sim\n'
'                         + 0.6 * maccs_cos + 0.6 * maccs_tan + 0.80 * mass_score - 0.04 * elem_dist)\n'
'                full_candidates.append(dict(molecule_id=str(molecule_id), smiles=Chem.MolToSmiles(mol, isomericSmiles=False),\n'
'                                            fast_key=key, score=score, log_prob=log_prob, nb_ll=nb_ll, cos_sim=cos_sim,\n'
'                                            tan_sim=tan_sim, maccs_cos=maccs_cos, maccs_tan=maccs_tan,\n'
"                                            elem_dist=elem_dist, ppm_err=ppm, source='spec2smiles'))\n"
'\n'
'            # Fast tautomer deduplication: top 12 use canon_ik14, ranks 13-25 use fast_key\n'
'            canon_seen, rank = set(), 0\n'
"            for candidate in sorted(full_candidates, key=lambda x: -x['score']):\n"
"                ik14 = evalkit.canon_ik14(candidate['smiles']) if rank < 12 else (candidate.get('fast_key') or evalkit.fast_key(candidate['smiles']))\n"
'                if not ik14 or ik14 in canon_seen:\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'                    continue\n'
'                canon_seen.add(ik14)\n'
'                rank += 1\n'
'                candidate.update(rank=rank, ik14=ik14)\n'
'                rows.append(candidate)\n'
'                if rank >= top_k:\n'
'                    break\n'
'\n'
'    return pd.DataFrame(rows, columns=COLUMNS).astype({\n'
"        'rank': 'int64', 'score': 'float64', 'log_prob': 'float64',\n"
"        'nb_ll': 'float64', 'cos_sim': 'float64', 'tan_sim': 'float64',\n"
"        'maccs_cos': 'float64', 'maccs_tan': 'float64',\n"
"        'elem_dist': 'float64', 'ppm_err': 'float64'\n"
'    })\n'
'\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
module_path = WORK / 'spec2smiles_infer.py'
module_path.write_text('', encoding='utf-8')

In [ ]:
fragment = (
'"""Stage 08 inference runtime, embedded verbatim into the generated notebook."""\n'
'import contextlib\n'
'import gc\n'
'import hashlib\n'
'import importlib.util\n'
'import json\n'
'import math\n'
'import os\n'
'from pathlib import Path\n'
'import random\n'
'import re\n'
'import sys\n'
'import time\n'
'from collections import Counter\n'
'from functools import lru_cache\n'
'\n'
'import numpy as np\n'
'import pandas as pd\n'
'import pyarrow as pa\n'
'import pyarrow.parquet as pq\n'
'import psutil\n'
'import torch\n'
'from rdkit import Chem, RDLogger\n'
'from rdkit.Chem import rdMolDescriptors\n'
'from rdkit.Chem.MolStandardize import rdMolStandardize\n'
'\n'
"ELEMENTS = ['C', 'H', 'N', 'O', 'S', 'P', 'F', 'Cl', 'Br', 'I']\n"
"EXTRA = ['Na', 'K', 'Si', 'B', 'Se']\n"
'PT = Chem.GetPeriodicTable()\n'
'MASS = {e: PT.GetMostCommonIsotopeMass(e) for e in ELEMENTS + EXTRA}\n'
'NOMINAL = {e: round(MASS[e]) for e in MASS}\n'
"ADDUCTS = {'[M+H]+': (1, 1.007276), '[M-H]-': (1, -1.007276),\n"
"    '[M+Na]+': (1, 22.989218), '[M+K]+': (1, 38.963158),\n"
"    '[M+NH4]+': (1, 18.033823), '[M+Cl]-': (1, 34.969402),\n"
"    '[M+2H]2+': (2, 2.014552), '[M-2H]2-': (2, -2.014552),\n"
"    '[M+H-H2O]+': (1, -17.003289), '[M-H2O+H]+': (1, -17.003289),\n"
"    '[M-H2O-H]-': (1, -19.017841), '[M+FA-H]-': (1, 44.998201),\n"
"    '[M+CH2O2-H]-': (1, 44.998201), '[M+HAc-H]-': (1, 59.013851),\n"
"    '[M+2Na-H]+': (1, 44.971160), '[M]+': (1, -0.000549)}\n"
"RAW_COLS = ['molecule_id', 'smiles', 'ik14', 'log_prob', 'mean_log_prob',\n"
"            'count', 'formula_score', 'ppm_err', 'mass_relaxed']\n"
'\n'
'\n'
"def find_file(pattern, hint='', required=True, roots=None):\n"
"    roots = roots or ['/kaggle/input', '/kaggle/working']\n"
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
"    if pattern == 'evalkit.py':\n"
"        path = Path('/kaggle/working/evalkit.py')\n"
'        if not path.is_file() and required:\n'
"            raise FileNotFoundError(f'Missing {path}; modules are not imported from input datasets')\n"
'        return path if path.is_file() else None\n'
'    hits = [p for root in roots for p in Path(root).rglob(pattern) if p.is_file()]\n'
'    hits.sort(key=lambda p: (bool(hint and hint in str(p)), p.stat().st_mtime,\n'
'                             str(p)), reverse=True)\n'
'    if not hits and required:\n'
"        raise FileNotFoundError(f'Missing {pattern}; searched {roots}')\n"
'    return hits[0] if hits else None\n'
'\n'
'\n'
'def load_evalkit(work):\n'
"    path = Path(work) / 'evalkit.py'\n"
'    if not path.is_file():\n'
"        raise FileNotFoundError(f'Provide the Stage 02 module at {path}')\n"
"    spec = importlib.util.spec_from_file_location('evalkit', path)\n"
'    module = importlib.util.module_from_spec(spec)\n'
"    sys.modules['evalkit'] = module\n"
'    spec.loader.exec_module(module)\n'
'    assert Path(module.__file__).resolve() == path.resolve()\n'
'    return module\n'
'\n'
'\n'
'def seed_all(seed):\n'
'    random.seed(seed)\n'
'    np.random.seed(seed)\n'
'    torch.manual_seed(seed)\n'
'    torch.backends.cudnn.deterministic = True\n'
'    torch.backends.cudnn.benchmark = False\n'
'    torch.set_num_threads(1)\n'
"    RDLogger.DisableLog('rdApp.*')\n"
'\n'
'\n'
'def check_ram(limit=10e9):\n'
'    used = psutil.Process().memory_info().rss\n'
'    if used >= limit:\n'
"        raise MemoryError(f'Process RAM {used / 1e9:.2f} GB >= {limit / 1e9} GB')\n"
'    return used\n'
'\n'
'\n'
'class SchemaAdapter:\n'
"    aliases = {'molecule_id': ['molecule_id', 'mol_id', 'id'],\n"
"        'smiles': ['smiles', 'SMILES', 'canonical_smiles'],\n"
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
"        'mz': ['mz', 'mzs', 'peak_mz'], 'inten': ['inten', 'intensity', 'intensities'],\n"
"        'precursor_mz': ['precursor_mz', 'precursor_mass', 'parent_mz'],\n"
"        'neutral_mass': ['neutral_mass', 'exact_mass'], 'adduct': ['adduct', 'adduct_type'],\n"
"        'ce': ['ce', 'collision_energy'], 'instrument': ['instrument', 'instrument_type'],\n"
"        'polarity': ['polarity', 'ion_mode'], 'formula': ['formula', 'molecular_formula']}\n"
'\n'
'    def __init__(self, path, required):\n'
'        self.path = str(path)\n'
'        names = pq.ParquetFile(path).schema_arrow.names\n'
'        self.mapping = {key: next((v for v in opts if v in names), None)\n'
'                        for key, opts in self.aliases.items()}\n'
'        missing = [key for key in required if self.mapping.get(key) is None]\n'
'        if missing:\n'
"            raise ValueError(f'{path}: missing {missing}; available {names}')\n"
'\n'
'    def batches(self, keys, batch_size=4096):\n'
'        mapping = {self.mapping[k]: k for k in keys if self.mapping.get(k)}\n'
'        for batch in pq.ParquetFile(self.path).iter_batches(\n'
'                batch_size=batch_size, columns=list(mapping), use_threads=False):\n'
'            frame = batch.to_pandas().rename(columns=mapping)\n'
"            if 'molecule_id' in frame:\n"
'                if frame.molecule_id.isna().any():\n'
"                    raise ValueError('Null molecule_id')\n"
"                frame['molecule_id'] = frame.molecule_id.astype(str)\n"
'            yield frame\n'
'\n'
'\n'
'def parse_formula(formula):\n'
"    parts = re.findall(r'([A-Z][a-z]?)(\\d*)', str(formula))\n"
"    if ''.join(e + n for e, n in parts) != str(formula):\n"
'        return {}\n'
'    result = Counter()\n'
'    for e, n in parts:\n'
'        result[e] += int(n or 1)\n'
'    return dict(result)\n'
'\n'
'\n'
'def formula_text(counts):\n'
"    order = ['C', 'H'] + sorted(set(counts) - {'C', 'H'})\n"
"    return ''.join(e + (str(counts[e]) if counts[e] != 1 else '')\n"
'                   for e in order if counts.get(e, 0))\n'
'\n'
'\n'
'@lru_cache(maxsize=100000)\n'
'def smiles_formula(smiles):\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'    mol = Chem.MolFromSmiles(smiles)\n'
'    if mol is None:\n'
"        return ''\n"
'    mol = rdMolStandardize.Uncharger().uncharge(rdMolStandardize.FragmentParent(mol))\n'
'    return formula_text(parse_formula(rdMolDescriptors.CalcMolFormula(mol)))\n'
'\n'
'\n'
'def mass_hypotheses(group):\n'
'    masses, unknown = [], set()\n'
"    for row in group[['precursor_mz', 'adduct', 'polarity']].drop_duplicates().itertuples():\n"
'        adduct = str(row.adduct).strip()\n'
'        if adduct not in ADDUCTS:\n'
'            unknown.add(adduct)\n'
"            neg = str(row.polarity).lower() in ('negative', '-', '-1')\n"
"            choices = ['[M-H]-', '[M+Cl]-'] if neg else ['[M+H]+', '[M+Na]+', '[M+NH4]+']\n"
'        else:\n'
'            choices = [adduct]\n'
'        for name in choices:\n'
'            charge, shift = ADDUCTS[name]\n'
'            mass = float(row.precursor_mz) * charge - shift\n'
'            if math.isfinite(mass) and mass > 0:\n'
'                masses.append((mass, name))\n'
'    if not masses:\n'
"        raise ValueError('No finite positive precursor-derived neutral mass')\n"
'    # Each adduct gives one robust mass estimate across replicate spectra.\n'
'    return [(float(np.median([m for m, a in masses if a == name])), name)\n'
'            for name in sorted({a for _, a in masses})], sorted(unknown)\n'
'\n'
'\n'
'def build_formula_stats(train_adapter, split_map, db_path, cfg):\n'
'    prior, seen, counts = Counter(), set(), []\n'
"    for frame in train_adapter.batches(['molecule_id', 'smiles']):\n"
"        frame = frame.loc[frame.molecule_id.map(split_map).eq('train')]\n"
"        for row in frame.drop_duplicates('molecule_id').itertuples():\n"
'            if row.molecule_id in seen:\n'
'                continue\n'
'            seen.add(row.molecule_id)\n'
'            f = smiles_formula(str(row.smiles))\n'
'            d = parse_formula(f)\n'
'            if d and not (set(d) - set(MASS)):\n'
'                prior[f] += 1\n'
'                counts.append(d)\n'
"        check_ram(cfg['parent_ram_gb'] * 1e9)\n"
'    if not counts:\n'
"        raise ValueError('No train-only formulas usable for chemical bounds')\n"
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'    allowed = ELEMENTS + [e for e in EXTRA if any(d.get(e, 0) for d in counts)]\n'
"    ratios = {e: np.percentile([d.get(e, 0) / max(d.get('C', 0), 1)\n"
'                               for d in counts], [1, 99]).tolist() for e in allowed}\n'
'    # Explicit finite search domain; no prediction-dependent pruning or time truncation.\n'
'    bounds = {e: max(d.get(e, 0) for d in counts) for e in allowed}\n'
"    bounds.update(cfg.get('formula_max_counts', {}))\n"
'    if db_path:\n'
'        adapter = SchemaAdapter(db_path, [])\n'
"        key = 'formula' if adapter.mapping['formula'] else 'smiles'\n"
'        if not adapter.mapping[key]:\n'
"            raise ValueError('DB needs a formula or smiles column')\n"
'        for frame in adapter.batches([key]):\n'
'            values = frame[key].dropna().astype(str).value_counts()\n'
'            for value, freq in values.items():\n'
"                f = formula_text(parse_formula(value)) if key == 'formula' else smiles_formula(value)\n"
'                if f:\n'
'                    prior[f] += int(freq)\n'
"            check_ram(cfg['parent_ram_gb'] * 1e9)\n"
'    return dict(prior=dict(prior), allowed=allowed, ratios=ratios, bounds=bounds,\n'
"                prior_definition='one count per train molecule plus each optional DB row')\n"
'\n'
'\n'
'def sane_formula(d, stats):\n'
"    c = d.get('C', 0)\n"
'    if c < 1:\n'
'        return False\n'
"    h, n = d.get('H', 0), d.get('N', 0)\n"
"    hal = sum(d.get(e, 0) for e in ('F', 'Cl', 'Br', 'I', 'Na', 'K'))\n"
"    dbe2 = 2 + 2 * (c + d.get('Si', 0)) + n + d.get('P', 0) + d.get('B', 0) - h - hal\n"
'    if dbe2 < 0 or dbe2 % 2:\n'
'        return False\n'
'    # Generalized closed-shell nitrogen parity for trivalent P/B and monovalent metals.\n'
'    nominal = sum(NOMINAL[e] * v for e, v in d.items())\n'
'    if nominal % 2 != n % 2:\n'
'        return False\n'
'    return all(lo - 1e-9 <= d.get(e, 0) / c <= hi + 1e-9\n'
"               for e, (lo, hi) in stats['ratios'].items())\n"
'\n'
'\n'
'def enumerate_formulas(mass, stats, ppm):\n'
'    """Exhaustive bounded heavy-element search; solve integer H at each leaf."""\n'
'    tolerance = mass * ppm / 1e6\n'
"    bounds, ratios = stats['bounds'], stats['ratios']\n"
"    elements = sorted((e for e in stats['allowed'] if e not in ('C', 'H')),\n"
'                      key=lambda e: MASS[e], reverse=True)\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
"    for c in range(1, min(bounds['C'], int((mass + tolerance) / 12)) + 1):\n"
"        d = {'C': c}\n"
'        ranges = {e: (max(0, math.ceil(ratios[e][0] * c - 1e-9)),\n'
'                      min(bounds[e], math.floor(ratios[e][1] * c + 1e-9)))\n'
"                  for e in elements + ['H']}\n"
'        if any(lo > hi for lo, hi in ranges.values()):\n'
'            continue\n'
'        suffix_min, suffix_max = [0.] * (len(elements) + 1), [0.] * (len(elements) + 1)\n'
"        suffix_min[-1], suffix_max[-1] = [x * MASS['H'] for x in ranges['H']]\n"
'        for i in range(len(elements) - 1, -1, -1):\n'
'            e = elements[i]\n'
'            suffix_min[i] = suffix_min[i+1] + ranges[e][0] * MASS[e]\n'
'            suffix_max[i] = suffix_max[i+1] + ranges[e][1] * MASS[e]\n'
'\n'
'        def walk(i, remaining):\n'
'            if remaining + tolerance < suffix_min[i] or remaining - tolerance > suffix_max[i]:\n'
'                return\n'
'            if i == len(elements):\n'
"                lo = max(ranges['H'][0], math.ceil((remaining - tolerance) / MASS['H']))\n"
"                hi = min(ranges['H'][1], math.floor((remaining + tolerance) / MASS['H']))\n"
'                for h in range(lo, hi + 1):\n'
"                    d['H'] = h\n"
'                    if sane_formula(d, stats):\n'
'                        yield formula_text(d)\n'
'                return\n'
'            e = elements[i]\n'
'            lo = max(ranges[e][0], math.ceil((remaining - tolerance - suffix_max[i+1]) / MASS[e]))\n'
'            hi = min(ranges[e][1], math.floor((remaining + tolerance - suffix_min[i+1]) / MASS[e]))\n'
'            for value in range(lo, hi + 1):\n'
'                d[e] = value\n'
'                yield from walk(i+1, remaining - value * MASS[e])\n'
'            d.pop(e, None)\n'
'        yield from walk(0, mass - c * 12)\n'
'\n'
'\n'
'def rank_formulas(hypotheses, pred, stats, cfg):\n'
'    scores = {}\n'
"    order = cfg['elements_order']\n"
'    pred = np.asarray(pred, dtype=float)\n'
'    for mass, adduct in hypotheses:\n'
"        for formula in enumerate_formulas(mass, stats, cfg['ppm']):\n"
'            d = parse_formula(formula)\n'
'            actual = np.array([d.get(e, 0) for e in order])\n'
'            count_loss = np.mean(np.abs(actual - pred) / np.sqrt(pred + 1))\n'
'            presence_loss = np.mean(np.abs((actual > 0) - (1 - np.exp(-pred))))\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
"            prior = math.log1p(stats['prior'].get(formula, 0))\n"
"            score = -count_loss - cfg['presence_weight'] * presence_loss + cfg['prior_weight'] * prior\n"
"            if formula not in scores or score > scores[formula]['score']:\n"
'                scores[formula] = dict(formula=formula, score=float(score), mass=mass, adduct=adduct)\n'
"    return sorted(scores.values(), key=lambda r: (-r['score'], r['formula']))\n"
'\n'
'\n'
'@lru_cache(maxsize=100000)\n'
'def canonical_record(smiles):\n'
'    mol = Chem.MolFromSmiles(smiles)\n'
'    if mol is None or not mol.GetNumAtoms():\n'
'        return None\n'
'    # Strict final deduplication: limited enumeration raises on exhaustion; no plain-key fallback.\n'
'    mol = rdMolStandardize.Uncharger().uncharge(rdMolStandardize.FragmentParent(mol))\n'
'    te = rdMolStandardize.TautomerEnumerator()\n'
'    te.SetMaxTautomers(1000)\n'
'    te.SetMaxTransforms(1000)\n'
'    enumeration = te.Enumerate(mol)\n'
'    if enumeration.status != rdMolStandardize.TautomerEnumeratorStatus.Completed:\n'
"        return ('canonicalization_limit',)\n"
'    canon = te.PickCanonical(enumeration)\n'
'    key = Chem.MolToInchiKey(canon)[:14]\n'
'    if not key:\n'
'        return None\n'
'    return (Chem.MolToSmiles(canon, isomericSmiles=False), key,\n'
'            float(rdMolDescriptors.CalcExactMolWt(canon)),\n'
'            formula_text(parse_formula(rdMolDescriptors.CalcMolFormula(canon))))\n'
'\n'
'\n'
'def postprocess(mid, paths, hypotheses, ranked, cfg):\n'
'    valid, limited, records = 0, 0, []\n'
"    formula_scores = {r['formula']: r['score'] for r in ranked}\n"
'    for smi, logp, method in paths:\n'
'        parsed = Chem.MolFromSmiles(smi) if smi else None\n'
'        if parsed is None or not parsed.GetNumAtoms():\n'
'            continue\n'
'        valid += 1\n'
'        record = canonical_record(smi)\n'
"        if record and record[0] == 'canonicalization_limit':\n"
'            limited += 1\n'
'            continue\n'
'        if record is None:\n'
'            continue\n'
'        canonical, key, mass, formula = record\n'
'        ppm = min(abs(mass - m) / m * 1e6 for m, _ in hypotheses)\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'        records.append(dict(molecule_id=mid, smiles=canonical, ik14=key,\n'
'            log_prob=logp, ppm_err=ppm, formula_score=formula_scores.get(formula,\n'
'                min(formula_scores.values(), default=0.) - 1.)))\n'
"    strict = [r for r in records if r['ppm_err'] <= cfg['ppm']]\n"
'    relaxed = not strict\n'
"    kept = strict or [r for r in records if r['ppm_err'] <= cfg['relax_ppm']]\n"
'    output = []\n'
"    for key in sorted({r['ik14'] for r in kept}):\n"
"        hits = [r for r in kept if r['ik14'] == key]\n"
"        best = max(hits, key=lambda r: r['log_prob']).copy()\n"
"        best.update(count=len(hits), mean_log_prob=float(np.mean([r['log_prob'] for r in hits])),\n"
'                    mass_relaxed=relaxed)\n'
'        output.append(best)\n'
'    diag = dict(molecule_id=mid, attempted=len(paths), valid=valid,\n'
'        canonicalization_limited=limited, strict_mass_hits=len(strict), kept_hits=len(kept),\n'
'        unique=len(output), no_strict_survivor=relaxed, no_survivor=not output,\n'
"        formula_count=len(ranked), formula_top3=[r['formula'] for r in ranked[:3]])\n"
'    return output, diag\n'
'\n'
'\n'
'def autocast(device):\n'
"    return torch.autocast('cuda', dtype=torch.float16) if device.type == 'cuda' else contextlib.nullcontext()\n"
'\n'
'\n'
'@torch.inference_mode()\n'
'def encode_group(model, group, device):\n'
'    """Average encoded memory at each valid position over all spectra, weighted by sqrt(peaks)."""\n'
'    import spec2smiles as s2s\n'
'    from spec2fp import collate_spectra\n'
'    memory_sum = weight_sum = pooled_sum = cond_sum = None\n'
'    total = 0.\n'
'    for row in group.itertuples(index=False):\n'
'        item = s2s.spectrum_item(row, model.cfg)\n'
"        batch = collate_spectra([item], model.cfg['top_k_peaks'], 2 * model.cfg['mz_dim'] + 1)\n"
'        batch = {k: v.to(device) for k, v in batch.items()}\n'
'        with autocast(device):\n'
'            memory, mask, pooled, cond = model.encode(**batch)\n'
"        weight = math.sqrt(max(item['n_peaks'], 1))\n"
'        token_weight = (~mask).float().unsqueeze(-1) * weight\n'
'        weighted = memory.float() * token_weight\n'
'        memory_sum = weighted if memory_sum is None else memory_sum + weighted\n'
'        weight_sum = token_weight if weight_sum is None else weight_sum + token_weight\n'
'        pooled_sum = pooled.float() * weight if pooled_sum is None else pooled_sum + pooled.float() * weight\n'
'        cond_sum = cond.float() * weight if cond_sum is None else cond_sum + cond.float() * weight\n'
'        total += weight\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'    hypotheses, _ = mass_hypotheses(group)\n'
"    mass = float(group.neutral_mass.median()) if 'neutral_mass' in group else hypotheses[0][0]\n"
'    if not math.isfinite(mass) or mass <= 0:\n'
'        mass = hypotheses[0][0]\n'
'    with autocast(device):\n'
'        return model.conditioned_memory(memory_sum / weight_sum.clamp_min(1),\n'
'            weight_sum.squeeze(-1).eq(0), torch.tensor([mass], device=device),\n'
'            pooled_sum / total, cond_sum / total)\n'
'\n'
'\n'
'@torch.inference_mode()\n'
'def decode_beam(model, memory, mask, cfg, prefix=None):\n'
'    """Full batched beam without unsafe early stopping; archive all completed paths."""\n'
'    b, length, dim = memory.shape\n'
"    k, device = cfg['beam'], memory.device\n"
'    mem = memory[:, None].expand(-1, k, -1, -1).reshape(b*k, length, dim)\n'
'    masks = mask[:, None].expand(-1, k, -1).reshape(b*k, length)\n'
'    prefix = prefix or [1]\n'
'    seq = torch.tensor(prefix, device=device)[None, None].expand(b, k, -1).clone()\n'
'    scores = torch.full((b, k), -torch.inf, device=device)\n'
'    scores[:, 0] = 0\n'
'    completed = [[] for _ in range(b)]\n'
"    for step in range(len(prefix), cfg['max_length']):\n"
'        with autocast(device):\n'
'            logits = model.decode(seq.reshape(b*k, -1), mem, masks)[:, -1].float()\n'
'        logp = logits.log_softmax(-1).reshape(b, k, -1)\n'
'        logp[:, :, [0, 1, 3]] = -torch.inf\n'
'        if step == len(prefix):\n'
'            logp[:, :, 2] = -torch.inf\n'
'        combined = logp + scores[:, :, None]\n'
'        eos = combined[:, :, 2].cpu().tolist()\n'
'        seq_cpu = seq.cpu().tolist()\n'
'        for i in range(b):\n'
'            for j, score in enumerate(eos[i]):\n'
'                if math.isfinite(score):\n'
'                    ids = seq_cpu[i][j][len(prefix):] + [2]\n'
"                    completed[i].append((ids, score / len(ids) ** cfg['length_penalty']))\n"
"            completed[i] = sorted(completed[i], key=lambda p: -p[1])[:cfg['num_return']]\n"
'        combined[:, :, 2] = -torch.inf\n'
'        scores, indices = combined.flatten(1).topk(k, dim=1)\n'
'        parents, tokens = indices // logp.shape[-1], indices % logp.shape[-1]\n'
'        seq = torch.cat((seq.gather(1, parents[:, :, None].expand(-1, -1, step)), tokens[:, :, None]), 2)\n'
'    return completed\n'
'\n'
'\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'@torch.inference_mode()\n'
'def decode_sample(model, memory, mask, cfg, prefix=None):\n'
'    b, length, dim = memory.shape\n'
"    n, device = cfg['samples'], memory.device\n"
'    mem = memory[:, None].expand(-1, n, -1, -1).reshape(b*n, length, dim)\n'
'    masks = mask[:, None].expand(-1, n, -1).reshape(b*n, length)\n'
'    prefix = prefix or [1]\n'
'    seq = torch.tensor(prefix, device=device)[None].expand(b*n, -1).clone()\n'
'    scores = torch.zeros(b*n, device=device)\n'
'    lengths = torch.zeros(b*n, device=device)\n'
'    done = torch.zeros(b*n, dtype=torch.bool, device=device)\n'
"    for step in range(len(prefix), cfg['max_length']):\n"
'        with autocast(device):\n'
'            logits = model.decode(seq, mem, masks)[:, -1].float()\n'
'        original = logits.log_softmax(-1)\n'
'        logits[:, [0, 1, 3]] = -torch.inf\n'
'        if step == len(prefix):\n'
'            logits[:, 2] = -torch.inf\n'
"        values, indices = (logits / cfg['temperature']).sort(descending=True)\n"
'        probs = values.softmax(-1)\n'
"        values[(probs.cumsum(-1) - probs) >= cfg['top_p']] = -torch.inf\n"
'        chosen = torch.multinomial(values.softmax(-1), 1)\n'
'        token = indices.gather(1, chosen).squeeze(1)\n'
'        scores += original.gather(1, token[:, None]).squeeze(1) * (~done)\n'
'        lengths += ~done\n'
'        token = torch.where(done, 2, token)\n'
'        seq = torch.cat((seq, token[:, None]), 1)\n'
'        done |= token.eq(2)\n'
'        if done.all():\n'
'            break\n'
'    result = [[] for _ in range(b)]\n'
'    for i, (ids, score, size, finished) in enumerate(zip(seq.cpu().tolist(),\n'
'            scores.cpu().tolist(), lengths.cpu().tolist(), done.cpu().tolist())):\n'
'        # Unfinished draws remain explicit invalid attempts in diagnostics.\n'
'        output = ids[len(prefix):] if finished else []\n'
"        result[i // n].append((output, score / max(size, 1) ** cfg['length_penalty']))\n"
'    return result\n'
'\n'
'\n'
'def prediction_map(paths, name, expected, cfg):\n'
"    ids = pd.read_parquet(paths[f'ids_{name}'], columns=['molecule_id']).molecule_id.astype(str)\n"
"    values = np.load(paths[f'elem_{name}'], mmap_mode='r')\n"
"    if ids.duplicated().any() or values.shape != (len(ids), len(cfg['elements_order'])):\n"
"        raise ValueError(f'{name}: element array/ID shape mismatch or duplicated IDs')\n"
'    missing = set(expected) - set(ids)\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'    if missing:\n'
"        raise ValueError(f'{name}: {len(missing)} missing element predictions')\n"
'    if not np.isfinite(values).all() or (values < 0).any():\n'
"        raise ValueError(f'{name}: invalid predicted element counts')\n"
'    wanted = set(expected)\n'
'    return {mid: np.array(values[i], copy=True) for i, mid in enumerate(ids) if mid in wanted}\n'
'\n'
'\n'
'def run_worker(job_path, shard):\n'
'    job = json.loads(Path(job_path).read_text())\n'
"    cfg, paths = job['cfg'], job['paths']\n"
"    seed_all(cfg['seed'] + shard)\n"
"    load_evalkit(cfg['work'])\n"
'    import spec2smiles as s2s\n'
"    device = torch.device('cpu' if cfg['SMOKE_TEST'] else 'cuda:0')\n"
"    if device.type == 'cuda':\n"
'        total = torch.cuda.get_device_properties(0).total_memory\n'
"        torch.cuda.set_per_process_memory_fraction(min(.95, cfg['gpu_memory_gb'] * 1e9 / total))\n"
'        torch.cuda.reset_peak_memory_stats()\n'
"    model_cfg = json.loads(Path(paths['config']).read_text())\n"
'    model = s2s.Spec2SmilesModel(model_cfg)\n'
"    checkpoint = torch.load(paths['model'], map_location='cpu', weights_only=False)\n"
"    state = checkpoint.get('ema_state_dict', checkpoint.get('model_state_dict', checkpoint))\n"
"    model.load_state_dict({k.removeprefix('module.'): v for k, v in state.items()}, strict=True)\n"
'    del checkpoint, state\n'
'    model.to(device).eval()\n'
"    tokenizer = s2s.SmilesTokenizer.load(paths['vocab'])\n"
"    if cfg['max_length'] > min(tokenizer.max_length, model.positions.shape[0]):\n"
"        raise ValueError('Configured sequence length exceeds trained positional/vocabulary limit')\n"
"    if model_cfg.get('formula_prefix', False):\n"
"        raise ValueError('This runtime targets Stage 07 v2; formula-prefix checkpoints need their trained adapter')\n"
"    print('Conditioning: native predicted elements; formula prefix/no-formula token unsupported', flush=True)\n"
"    stats = json.loads(Path(paths['formula_stats']).read_text())\n"
"    out = Path(cfg['out'])\n"
"    batch_size = cfg['batch_size']\n"
"    for name in ('val_model', 'val_rank', 'test'):\n"
"        parts = [Path(p) for p in job['parts'][name][str(shard)]]\n"
"        expected = job['ids'][name][shard::2]\n"
'        predictions = prediction_map(paths, name, expected, cfg)\n'
'        for part in parts:\n'
"            seed_all(cfg['seed'] + shard + int(part.stem) * 101 +\n"
"                     {'val_model': 0, 'val_rank': 100000, 'test': 200000}[name])\n"
"            stem = out / f'{name}_shard{shard}_{part.stem}'\n"
"            complete = stem.with_suffix('.done.json')\n"
"            if cfg['resume'] and complete.exists():\n"
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
"                if json.loads(complete.read_text())['run_hash'] == job['run_hash']:\n"
'                    continue\n'
'            frame = pd.read_parquet(part)\n'
"            groups = list(frame.groupby('molecule_id', sort=False))\n"
'            rows, diagnostics, formula_rows, index = [], [], [], 0\n'
'            started = time.perf_counter()\n'
'            while index < len(groups):\n'
'                chunk = groups[index:index + batch_size]\n'
'                attempt_rng = torch.get_rng_state()\n'
"                cuda_rng = torch.cuda.get_rng_state() if device.type == 'cuda' else None\n"
'                try:\n'
'                    generated = generate_chunk(model, tokenizer, chunk, device, cfg)\n'
'                except torch.OutOfMemoryError:\n'
'                    if batch_size == 1:\n'
"                        raise RuntimeError('OOM at batch=1; beam/sample counts preserved. Use a larger GPU.')\n"
'                    batch_size = max(1, batch_size // 2)\n'
'                    gc.collect()\n'
'                    torch.cuda.empty_cache()\n'
'                    torch.set_rng_state(attempt_rng)\n'
'                    if cuda_rng is not None:\n'
'                        torch.cuda.set_rng_state(cuda_rng)\n'
'                    print(f\'OOM: retry batch={batch_size}, beam={cfg["beam"]}\', flush=True)\n'
'                    continue\n'
'                for (mid, group), paths_generated in zip(chunk, generated):\n'
'                    hypotheses, unknown = mass_hypotheses(group)\n'
'                    ranked = rank_formulas(hypotheses, predictions[mid], stats, cfg)\n'
'                    candidates, diag = postprocess(mid, paths_generated, hypotheses, ranked, cfg)\n'
'                    diag.update(split=name, unknown_adducts=unknown)\n'
'                    rows.extend(candidates)\n'
'                    diagnostics.append(diag)\n'
'                    formula_rows.extend(dict(molecule_id=mid, rank=i+1, **r) for i, r in enumerate(ranked[:3]))\n'
'                index += len(chunk)\n'
"                check_ram(cfg['worker_ram_gb'] * 1e9)\n"
"                print(f'{name} shard={shard} {part.stem}: {index}/{len(groups)}', flush=True)\n"
"            pd.DataFrame(rows, columns=RAW_COLS).to_parquet(stem.with_suffix('.parquet'), index=False)\n"
"            pd.DataFrame(diagnostics).to_parquet(stem.with_suffix('.diag.parquet'), index=False)\n"
"            pd.DataFrame(formula_rows, columns=['molecule_id', 'rank', 'formula', 'score', 'mass', 'adduct']).to_parquet(stem.with_suffix('.formula.parquet'), index=False)\n"
"            peak = torch.cuda.max_memory_reserved() / 1e9 if device.type == 'cuda' else 0.\n"
'            if peak >= 12:\n'
"                raise MemoryError(f'GPU memory budget violated: {peak:.2f} GB')\n"
"            complete.write_text(json.dumps(dict(run_hash=job['run_hash'], queries=len(groups),\n"
'                seconds=time.perf_counter()-started, peak_gpu_gb=peak, batch_size=batch_size)))\n'
"            print(f'{name} shard={shard} {part.stem}: {len(groups)} queries; peak={peak:.2f} GB', flush=True)\n"
'            del frame, groups, rows, diagnostics, formula_rows\n'
'            gc.collect()\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
fragment = (
'\n'
'\n'
'def generate_chunk(model, tokenizer, chunk, device, cfg):\n'
'    encoded = [encode_group(model, group, device) for _, group in chunk]\n'
'    memory = torch.cat([m for m, _ in encoded])\n'
'    mask = torch.cat([m for _, m in encoded])\n'
'    beam = decode_beam(model, memory, mask, cfg)\n'
'    sample = decode_sample(model, memory, mask, cfg)\n'
"    return [[(tokenizer.decode(ids) if ids else '', score, method)\n"
"             for method, paths in [('beam', b), ('sample', s)] for ids, score in paths]\n"
'            for b, s in zip(beam, sample)]\n'
'\n'
'\n'
'def scored(frame, alpha, formula_weight=0.):\n'
'    frame = frame.copy()\n'
"    frame['score'] = frame.log_prob + alpha * np.log(frame['count']) + formula_weight * frame.formula_score\n"
"    frame['source'] = 'gen'\n"
"    return frame.sort_values(['molecule_id', 'score', 'ik14'], ascending=[True, False, True])\n"
'\n'
'\n'
'def ranking_metrics(frame, truth):\n'
'    ranks = {}\n'
"    for mid, group in frame.groupby('molecule_id', sort=False):\n"
'        keys = group.ik14.drop_duplicates().head(25).tolist()\n'
'        if truth.get(mid) in keys:\n'
'            ranks[mid] = keys.index(truth[mid]) + 1\n'
'    n = len(truth)\n'
'    return dict(queries=n, mrr25=sum(1/r for r in ranks.values()) / max(n, 1),\n'
"                **{f'top{k}': sum(r <= k for r in ranks.values()) / max(n, 1) for k in (1, 5, 25)})\n"
'\n'
'\n'
'def sha256(path):\n'
'    digest = hashlib.sha256()\n'
"    with Path(path).open('rb') as stream:\n"
"        for block in iter(lambda: stream.read(8*1024*1024), b''):\n"
'            digest.update(block)\n'
'    return digest.hexdigest()\n'
'\n'
'\n'
"if __name__ == '__main__':\n"
'    run_worker(sys.argv[1], int(sys.argv[2]))\n'
)
with module_path.open('a', encoding='utf-8') as stream:
    stream.write(fragment)

In [ ]:
sys.path.insert(0, str(WORK))
importlib.invalidate_caches()
for name in ('spec2fp', 'spec2smiles', 'spec2smiles_infer'):
    sys.modules.pop(name, None)
import spec2smiles_infer as infer
infer.seed_all(CFG['seed'])
if not CFG['SMOKE_TEST']:
    evalkit_path = infer.find_file('evalkit.py', roots=[str(WORK)])
    assert evalkit_path.resolve() == (WORK / 'evalkit.py').resolve()
    evalkit = infer.load_evalkit(WORK)
    print('Read-only evalkit:', evalkit.__file__)

## Synthetic CPU smoke test
Exercises both full-width decoders, invalid/unfinished handling, mass rejection,
conditional relaxation, canonical aggregation, formula ranking and metrics. It does
not claim to validate the real trained checkpoint or GPU throughput.

In [ ]:
if CFG['SMOKE_TEST']:
    class SmokeDecoder(torch.nn.Module):
        def decode(self, tokens, memory, mask):
            logits = torch.full((*tokens.shape, 8), -20.)
            target = [4, 4, 5, 2][min(tokens.shape[1]-1, 3)]
            logits[:, -1, target] = 10.
            return logits
    cfg = dict(CFG, max_length=8)
    memory, mask = torch.zeros(2, 4, 8), torch.zeros(2, 4, dtype=torch.bool)
    model = SmokeDecoder()
    beams = infer.decode_beam(model, memory, mask, cfg)
    samples = infer.decode_sample(model, memory, mask, cfg)
    assert len(beams) == 2 and all(len(s) == 50 for s in samples)
    assert all(s[0][0][:4] == [4, 4, 5, 2] for s in samples)
    mass = 46.041864812
    paths = [('CCO', -1., 'beam'), ('OCC', -2., 'sample'),
             ('not_smiles', -3., 'sample'), ('C', -4., 'sample')]
    rows, diag = infer.postprocess('q', paths, [(mass, '[M+H]+')], [], cfg)
    assert len(rows) == 1 and rows[0]['count'] == 2 and diag['valid'] == 3
    relaxed, info = infer.postprocess('q', paths, [(mass*(1+30e-6), 'x')], [], cfg)
    assert relaxed and info['no_strict_survivor']
    empty, info = infer.postprocess('q', paths, [(100., 'x')], [], cfg)
    assert not empty and info['no_survivor']
    stats = dict(allowed=['C','H','O'], bounds=dict(C=3,H=10,O=2),
        ratios=dict(C=[1,1], H=[0,4], O=[0,2]), prior={'C2H6O': 5})
    pred = np.array([2,6,0,1,0,0,0,0,0,0])
    ranked = infer.rank_formulas([(mass, 'x')], pred, stats, cfg)
    assert ranked[0]['formula'] == 'C2H6O'
    table = infer.scored(pd.DataFrame(rows), 0.25)
    assert infer.ranking_metrics(table, {'q': rows[0]['ik14'], 'missing': 'x'})['mrr25'] == .5
    table.to_parquet(OUT / 'smoke_candidates.parquet', index=False)
    print('CPU smoke checks passed; production pipeline skipped.')

## Discover inputs and verify schemas / split coverage

In [ ]:
if not CFG['SMOKE_TEST']:
    paths = {key: str(infer.find_file(name)) for key, name in dict(
        train='train_clean.parquet', test='test_clean.parquet', splits='splits.parquet',
        model='spec2smiles_model.pt').items()}
    for key, filename in [('vocab','smiles_vocab.json'),('config','spec2smiles_config.json')]:
        paths[key] = str(infer.find_file(filename, hint=str(Path(paths['model']).parent)))
    for name in ('val_model', 'val_rank', 'test'):
        paths[f'elem_{name}'] = str(infer.find_file(f'elem_pred_{name}.npy'))
        paths[f'ids_{name}'] = str(infer.find_file(f'fp_pred_ids_{name}.parquet',
            hint=str(Path(paths[f'elem_{name}']).parent)))
    db_path = infer.find_file('db.parquet', required=False)
    schema_path = infer.find_file('schema.json', required=False)
    if schema_path:
        schema_metadata = json.loads(schema_path.read_text())
        print('Loaded schema.json; normalized required columns verified below:', schema_path)
    required = ['molecule_id','mz','inten','precursor_mz','adduct','ce','instrument','polarity']
    train_adapter = infer.SchemaAdapter(paths['train'], required + ['smiles'])
    test_adapter = infer.SchemaAdapter(paths['test'], required)
    splits = pd.read_parquet(paths['splits'])
    assert {'molecule_id','split3'} <= set(splits)
    splits['molecule_id'] = splits.molecule_id.astype(str)
    assert not splits.molecule_id.duplicated().any()
    assert set(splits.split3) == {'train','val_model','val_rank'}
    split_map = splits.set_index('molecule_id').split3
    train_ids = set()
    for frame in train_adapter.batches(['molecule_id']):
        train_ids.update(frame.molecule_id)
    assert train_ids == set(split_map.index), 'Split/train molecule coverage mismatch'
    test_ids = set()
    for frame in test_adapter.batches(['molecule_id']):
        test_ids.update(frame.molecule_id)
    assert not (test_ids & train_ids), 'Train/test molecule IDs overlap'
    ids = {name: sorted(splits.loc[splits.split3.eq(name), 'molecule_id'])
           for name in ('val_model','val_rank')}
    ids['test'] = sorted(test_ids)
    for name in ids:
        infer.prediction_map(paths, name, ids[name], CFG)
    saved_cfg = json.loads(Path(paths['config']).read_text())
    assert saved_cfg.get('elements_order', CFG['elements_order']) == CFG['elements_order']
    assert not saved_cfg.get('FULL_TRAIN', False), 'Full-train checkpoint contaminates validation'
    print({name: len(values) for name, values in ids.items()})

## Train-only formula priors and two query shards
Known adducts determine neutral masses. Unknown adducts are logged and use polarity
hypotheses. Filtering accepts the ppm window of any retained hypothesis. Formula
sanity includes nonnegative integer DBE, generalized nitrogen parity and train ratios.
Array rows are joined by their accompanying molecule IDs, never by parquet row order.

In [ ]:
if not CFG['SMOKE_TEST']:
    formula_stats = infer.build_formula_stats(train_adapter, split_map, db_path, CFG)
    paths['formula_stats'] = str(OUT / 'formula_stats.json')
    Path(paths['formula_stats']).write_text(json.dumps(formula_stats))
    print('Formula bounds:', formula_stats['bounds'])
    assignment = {mid: (name, i % 2, (i // 2) // CFG['queries_per_part'])
                  for name, values in ids.items() for i, mid in enumerate(values)}
    writers = {}
    parts = {name: {str(s): [] for s in (0,1)} for name in ids}
    try:
        for adapter in (train_adapter, test_adapter):
            for frame in adapter.batches(required + ['neutral_mass']):
                frame = frame.loc[frame.molecule_id.isin(assignment)].copy()
                frame['ce'] = pd.to_numeric(frame.ce, errors='coerce').fillna(0.)
                frame['instrument'] = frame.instrument.fillna('<unk>').astype(str)
                frame['adduct'] = frame.adduct.fillna('<unk>').astype(str)
                frame['polarity'] = frame.polarity.fillna('<unk>').astype(str)
                groups = frame.molecule_id.map(assignment)
                for key, group in frame.groupby(groups, sort=False):
                    name, shard, part = key
                    folder = CACHE / name / str(shard)
                    folder.mkdir(parents=True, exist_ok=True)
                    table = pa.Table.from_pandas(group, preserve_index=False)
                    if key not in writers:
                        part_path = folder / f'{part:05d}.parquet'
                        parts[name][str(shard)].append(str(part_path))
                        writers[key] = pq.ParquetWriter(part_path, table.schema, compression='zstd')
                    writers[key].write_table(table)
                infer.check_ram(CFG['parent_ram_gb'] * 1e9)
    finally:
        for writer in writers.values():
            writer.close()
    input_hashes = {key: infer.sha256(value) for key, value in paths.items()}
    source_hashes = {name: infer.sha256(WORK / name) for name in
                     ('spec2fp.py','spec2smiles.py','spec2smiles_infer.py','evalkit.py')}
    run_hash = hashlib.sha256(json.dumps(dict(cfg=CFG, inputs=input_hashes,
        sources=source_hashes), sort_keys=True).encode()).hexdigest()
    for name in parts:
        for shard in parts[name]:
            parts[name][shard].sort()
    job = dict(cfg=CFG, paths=paths, ids=ids, parts=parts, run_hash=run_hash)
    job_path = OUT / 'job.json'
    job_path.write_text(json.dumps(job))
    del assignment, writers, formula_stats
    gc.collect()

## Launch exactly two isolated inference workers
Two subprocesses receive disjoint molecule lists and independent CUDA contexts.
With one GPU they run sequentially on that GPU; normal Kaggle T4x2 runs concurrently.
Every 100-query part is checkpointed with a configuration/input/source hash. A matching
completed part resumes safely. No beam/sampling reductions or time-budget truncation.

In [ ]:
if not CFG['SMOKE_TEST']:
    gpu_count = torch.cuda.device_count()
    assert gpu_count >= 1, 'Production inference requires GPU; use SMOKE_TEST on CPU'
    def launch(shard):
        env = os.environ.copy()
        env.update(CUDA_VISIBLE_DEVICES=str(shard if gpu_count >= 2 else 0),
            OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', PYTHONUNBUFFERED='1',
            PYTHONHASHSEED=str(CFG['seed']))
        log = (OUT / f'worker_{shard}.log').open('w')
        process = subprocess.Popen([sys.executable, str(WORK / 'spec2smiles_infer.py'),
            str(job_path), str(shard)], env=env, stdout=log, stderr=subprocess.STDOUT)
        return process, log
    workers, launch_started = {}, time.perf_counter()
    workers[0] = launch(0)
    if gpu_count >= 2:
        workers[1] = launch(1)
    projected = False
    try:
        while True:
            if gpu_count == 1 and workers[0][0].poll() == 0 and 1 not in workers:
                workers[1] = launch(1)
            for shard, (proc, log) in workers.items():
                if proc.poll() not in (None, 0):
                    raise RuntimeError(f'Worker {shard} failed; inspect worker_{shard}.log')
            pilot = [OUT / f'val_model_shard{s}_00000.done.json' for s in (0,1)]
            if not projected and all(p.exists() for p in pilot):
                reports = [json.loads(p.read_text()) for p in pilot]
                rates = [r['queries']/max(r['seconds'], 1e-9) for r in reports]
                rate = sum(rates) if gpu_count >= 2 else 2 / sum(1/r for r in rates)
                projection = dict(pilot_queries=sum(r['queries'] for r in reports),
                    generator_test_seconds=len(ids['test'])/rate, gpu_count=gpu_count,
                    includes='encoding, generation, formula search, postprocessing, shard writes')
                (OUT / 'runtime_projection.json').write_text(json.dumps(projection, indent=2))
                print('Initial stage-10 generator projection:', projection, flush=True)
                projected = True
            completed = list(OUT.glob('*.done.json'))
            print('Completed query parts:', len(completed), 'elapsed min:',
                  round((time.perf_counter()-launch_started)/60, 1), flush=True)
            if len(workers) == 2 and all(p.poll() == 0 for p, _ in workers.values()):
                break
            time.sleep(20)
    finally:
        for proc, log in workers.values():
            if proc.poll() is None:
                proc.terminate()
                proc.wait()
            log.close()
    measured_inference_seconds = time.perf_counter() - launch_started

## Merge, tune alpha on val_model, and evaluate
Raw duplicate hits are retained across beam and sampling; max and mean length-normalized
model log-probabilities are recorded. `score = max_log_prob + alpha*log(count)` plus
an optional formula-score term. Alpha/formula weight use val_model only; val_rank is
evaluated afterward without tuning. All ranks use tautomer-canonical InChIKey14.
Canonicalization that exhausts its 1000-tautomer/transform guard is rejected and counted.
Validity is valid RDKit parses / returned beam hypotheses plus all sampling attempts;
unfinished samples count as invalid. Mass consistency is strict mass hits / valid parses.
Uniqueness is unique survivors / surviving raw hits, after the conditional 50-ppm relaxation.

In [ ]:
if not CFG['SMOKE_TEST']:
    raw, diagnostics, formula_tables = {}, {}, {}
    for name in ids:
        expected_parts = [OUT / f'{name}_shard{s}_{Path(part).stem}' for s in (0,1)
            for part in job['parts'][name][str(s)]]
        assert all(p.with_suffix('.done.json').exists() for p in expected_parts)
        raw[name] = pd.concat([pd.read_parquet(p.with_suffix('.parquet'))
                               for p in expected_parts], ignore_index=True)
        diagnostics[name] = pd.concat([pd.read_parquet(p.with_suffix('.diag.parquet'))
                                       for p in expected_parts], ignore_index=True)
        formula_tables[name] = pd.concat([pd.read_parquet(p.with_suffix('.formula.parquet'))
                                          for p in expected_parts], ignore_index=True)
        assert not diagnostics[name].molecule_id.duplicated().any()
        assert set(diagnostics[name].molecule_id) == set(ids[name])
        diagnostics[name].to_parquet(OUT / f'gen_diagnostics_{name}.parquet', index=False)
        formula_tables[name].to_parquet(OUT / f'formula_rank_{name}.parquet', index=False)
    truths, true_formulas = {}, {}
    label_sets = {name: {} for name in ('val_model','val_rank')}
    for frame in train_adapter.batches(['molecule_id','smiles']):
        for row in frame.itertuples():
            name = split_map[row.molecule_id]
            if name in label_sets:
                label_sets[name].setdefault(row.molecule_id, Counter()).update([str(row.smiles)])
    for name, labels in label_sets.items():
        truths[name], true_formulas[name] = {}, {}
        for mid, choices in tqdm(labels.items(), desc=f'Canonical truth {name}'):
            smi = sorted(choices, key=lambda s: (-choices[s], s))[0]
            result = infer.canonical_record(smi)
            assert result and len(result) == 4, f'Cannot canonicalize truth {mid}'
            truths[name][mid], true_formulas[name][mid] = result[1], result[3]
        assert set(truths[name]) == set(ids[name])
    trials = []
    for alpha in CFG['alpha_grid']:
        for weight in CFG['formula_weight_grid']:
            result = infer.ranking_metrics(infer.scored(raw['val_model'], alpha, weight), truths['val_model'])
            trials.append(dict(alpha=alpha, formula_weight=weight, **result))
    tuning = pd.DataFrame(trials).sort_values(['mrr25','alpha','formula_weight'],
                                            ascending=[False,True,True])
    best = tuning.iloc[0]
    tuning.to_csv(OUT / 'alpha_tuning_val_model.csv', index=False)
    print('Selected on val_model:', best.to_dict())

In [ ]:
if not CFG['SMOKE_TEST']:
    metrics, comparisons, final = {}, [], {}
    for name in ids:
        final[name] = infer.scored(raw[name], float(best.alpha), float(best.formula_weight))
        final[name].to_parquet(OUT / f'cand_gen_{name}.parquet', index=False)
        d = diagnostics[name]
        metrics[name] = dict(queries=len(ids[name]),
            validity_pct=100*int(d.valid.sum())/max(int(d.attempted.sum()),1),
            mass_consistency_pct=100*int(d.strict_mass_hits.sum())/max(int(d.valid.sum()),1),
            uniqueness= float(d.unique.sum())/max(int(d.kept_hits.sum()),1),
            canonicalization_limited=int(d.canonicalization_limited.sum()),
            relaxed_queries=int(d.no_strict_survivor.sum()), empty_queries=int(d.no_survivor.sum()))
        if name in truths:
            metrics[name].update(infer.ranking_metrics(final[name], truths[name]))
            ranks = formula_tables[name]
            matched = ranks.loc[ranks.formula.eq(ranks.molecule_id.map(true_formulas[name]))]
            metrics[name]['formula_top1'] = matched.loc[matched['rank'].le(1)].molecule_id.nunique()/len(ids[name])
            metrics[name]['formula_top3'] = matched.loc[matched['rank'].le(3)].molecule_id.nunique()/len(ids[name])
            comparisons.append(dict(split=name, method='gen', **metrics[name]))
            fp_path = infer.find_file(f'cand_fp_{name}.parquet', required=False)
            if fp_path:
                fp = pd.read_parquet(fp_path)
                fp['molecule_id'] = fp.molecule_id.astype(str)
                order, ascending = ('rank', True) if 'rank' in fp else ('score', False)
                fp = fp.sort_values(['molecule_id', order], ascending=[True, ascending])
                fp = fp.groupby('molecule_id', sort=False).head(25).copy()
                fp['ik14'] = fp.smiles.map(lambda s: infer.canonical_record(str(s)))
                fp = fp.loc[fp.ik14.map(lambda r: r is not None and len(r) == 4)].copy()
                fp['ik14'] = fp.ik14.map(lambda r: r[1])
                comparisons.append(dict(split=name, method='fp',
                    **infer.ranking_metrics(fp, truths[name])))
            else:
                print(f'FP comparison unavailable: cand_fp_{name}.parquet not attached')
    comparison = pd.DataFrame(comparisons)
    comparison.to_csv(OUT / 'gen_vs_fp_metrics.csv', index=False)
    print(comparison.to_string(index=False))
    print('Generator diagnostics:', json.dumps(metrics, indent=2))

## Submission and artifact checks
Generator outputs cannot mathematically guarantee a mass-consistent survivor for
every query. Both empty stages are flagged. If even 50 ppm yields no candidate,
the notebook saves all candidates/diagnostics and an explicitly incomplete submission,
then fails the final submission check. It never inserts aspirin, labels, or a
mass-inconsistent guess. Stage 10 can combine those queries with retrieval candidates.

In [ ]:
if not CFG['SMOKE_TEST']:
    prediction = {mid: ';'.join(group.head(25).smiles)
                  for mid, group in final['test'].groupby('molecule_id', sort=False)}
    submission = pd.DataFrame(dict(molecule_id=ids['test'],
        smiles=[prediction.get(mid, '') for mid in ids['test']]))
    submission.to_csv(OUT / 'submission_gen.csv', index=False)
    valid, errors, warnings, submission_stats = evalkit.validate_submission(submission, ids['test'])
    reports = {name: [json.loads(p.read_text()) for p in OUT.glob(f'{name}_shard*.done.json')
                     if json.loads(p.read_text())['run_hash'] == run_hash] for name in ids}
    timing = {name: dict(queries=sum(r['queries'] for r in rs),
        worker_seconds=sum(r['seconds'] for r in rs),
        peak_gpu_gb=max((r['peak_gpu_gb'] for r in rs), default=0.)) for name, rs in reports.items()}
    shard_costs = [sum(json.loads(p.read_text())['seconds'] for p in OUT.glob(f'test_shard{s}_*.done.json')
        if json.loads(p.read_text())['run_hash'] == run_hash) for s in (0,1)]
    generator_cost = max(shard_costs) if gpu_count >= 2 else sum(shard_costs)
    runtime = dict(measured_launch_wall_seconds=measured_inference_seconds,
        total_notebook_seconds=time.perf_counter()-START, per_split=timing,
        projected_stage10_generator_seconds=generator_cost,
        projected_stage10_generator_gpu_hours=timing['test']['worker_seconds']/3600,
        exceeds_stage10_generator_budget=generator_cost > CFG['stage10_generator_budget_s'],
        note='Worker-based estimate excludes other stage-10 models; resume may reuse prior timings')
    print('Measured runtime and stage-10 component cost:', json.dumps(runtime, indent=2))
    (OUT / 'gen_metrics.json').write_text(json.dumps(dict(splits=metrics,
        selected=dict(alpha=float(best.alpha), formula_weight=float(best.formula_weight)),
        runtime=runtime, submission_valid=bool(valid), errors=errors), indent=2))
    heldout_hash = hashlib.sha256('\n'.join(sorted(ids['val_model']+ids['val_rank'])).encode()).hexdigest()
    manifest = dict(stage='08_spec2smiles_infer', created=pd.Timestamp.now(tz='UTC').isoformat(),
        cfg=CFG, run_hash=run_hash, trained_on_split='train (upstream checkpoint)',
        heldout_ids_hash=heldout_hash, selected_on='val_model', reported_on='val_rank',
        conditioning='native predicted elements; formula prefix unsupported by Stage 07 v2',
        fp_spec=getattr(evalkit,'FP_SPEC',None), input_sha256=input_hashes,
        source_sha256=source_hashes, metrics=metrics, runtime=runtime,
        lib_versions=dict(torch=torch.__version__, rdkit=Chem.rdBase.rdkitVersion),
        submission_valid=bool(valid), files={p.name: dict(bytes=p.stat().st_size,
            sha256=infer.sha256(p)) for p in OUT.iterdir() if p.is_file() and p.name != 'manifest.json'})
    (OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
    for name in ids:
        frame = final[name]
        assert {'molecule_id','smiles','ik14','score','count','source'} <= set(frame)
        assert frame.source.eq('gen').all() and frame['count'].ge(1).all()
        assert not frame.duplicated(['molecule_id','ik14']).any()
        assert frame.ppm_err.le(CFG['relax_ppm']).all()
    assert valid, f'Incomplete generator submission; see diagnostics and fuse retrieval in Stage 10: {errors}'
    print('Stage 08 complete:', OUT)

## Run settings and interpretation
Use Kaggle **T4 ×2, Internet OFF**. Adjust `beam` / `num_return` together within
25–50, or increase `batch_size` for throughput; OOM retries halve only the batch.
`alpha_grid` and `formula_weight_grid` are selected exclusively on val_model.
`formula_max_counts` expands the empirical enumeration bounds explicitly.
Run times are workload-dependent; no real checkpoint/GPU benchmark is embedded.
The val_model score is a tuning score; val_rank is reported separately.

The cells above contain the complete embedded module source. The appendix below
prints it for archival inspection, as required by the project notebook rules.

In [ ]:
if not CFG['SMOKE_TEST']:
    for module_name in ('spec2fp.py', 'spec2smiles.py', 'spec2smiles_infer.py'):
        print(f'===== {module_name} =====')
        print((WORK / module_name).read_text(encoding='utf-8'))